# 03. Exploratory Parcel-Level Layer Analysis

This notebook decomposes network-level layer-wise encoding results into Schaefer parcels. Several sections are exploratory and were used to inspect parcel hotspots, within-network heterogeneity, and spatial gradients.

**GitHub-ready notebook:** outputs have been cleared and private paths have been replaced with placeholders.

In [ ]:
# Step 11A: Build Parcel-Level Encoding Table
#
# This cell converts the existing network-level layer-wise encoding NIfTI maps
# into a Schaefer parcel-level table. For each network, GPT-2 representation
# index, and parcel, it summarizes Raw+LLM, SRM+LLM, and SRM-minus-Raw encoding
# values. The resulting table is the foundation for parcel-level preferred-layer,
# within-network heterogeneity, and spatial-gradient analyses.

from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import image

pd.options.display.float_format = "{:.6f}".format
np.set_printoptions(suppress=True, precision=6)

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except Exception:
    HAS_TQDM = False

# =========================
# 1. Project setup
# =========================

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
PROJECT_DIR = BASE_DIR / "SNL_layerwise_encoding_fixed_srm50"
STEP10_DIR = PROJECT_DIR / "step10_layerwise_brain_mapping"
STEP10_NII_DIR = STEP10_DIR / "nii"

STEP11A_DIR = BASE_DIR / "parcel_layer_network_outputs" / "step11A_build_parcel_level_encoding_table"
CSV_DIR = STEP11A_DIR / "csv"
XLSX_DIR = STEP11A_DIR / "xlsx"
FIG_DIR = STEP11A_DIR / "figures"
JSON_DIR = STEP11A_DIR / "json"
for d in [STEP11A_DIR, CSV_DIR, XLSX_DIR, FIG_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ATLAS_PATH = Path(r"YOUR_SCHAEFER400_ATLAS_NIFTI_PATH")
LABEL_TXT_PATH = Path(r"YOUR_SCHAEFER400_LABEL_TXT_PATH")

NETWORK_NAMES = [
    "VisCent", "VisPeri",
    "SomMotA", "SomMotB",
    "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB",
    "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC",
    "DefaultA", "DefaultB", "DefaultC",
    "TempPar",
]

NETWORK_FAMILY_MAP = {
    "VisCent": "Visual",
    "VisPeri": "Visual",
    "SomMotA": "SomMot",
    "SomMotB": "SomMot",
    "DorsAttnA": "DorsAttn",
    "DorsAttnB": "DorsAttn",
    "SalVentAttnA": "SalVentAttn",
    "SalVentAttnB": "SalVentAttn",
    "LimbicA": "Limbic",
    "LimbicB": "Limbic",
    "ContA": "Control",
    "ContB": "Control",
    "ContC": "Control",
    "DefaultA": "Default",
    "DefaultB": "Default",
    "DefaultC": "Default",
    "TempPar": "TempPar",
}

LAYER_INDICES = list(range(49))  # 0 = embedding; 1-48 = transformer layers.
EPS = 1e-8

for p in [STEP10_NII_DIR, ATLAS_PATH, LABEL_TXT_PATH]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing required input: {p}")

start_time = time.time()
print("Step 11A: parcel-level encoding table")
print(f"Project directory : {PROJECT_DIR}")
print(f"Step 10 NIfTI dir : {STEP10_NII_DIR}")
print(f"Step 11A output   : {STEP11A_DIR}")

# =========================
# 2. Load Schaefer labels
# =========================

def parse_schaefer_label_txt(label_txt_path: Path) -> pd.DataFrame:
    rows = []
    with open(label_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            parcel_id = int(parts[0])
            parcel_label = parts[1]
            label_parts = parcel_label.split("_")
            hemisphere = label_parts[1] if len(label_parts) > 1 else "unknown"
            schaefer_network = label_parts[2] if len(label_parts) > 2 else "unknown"
            parcel_short_name = "_".join(label_parts[3:]) if len(label_parts) > 3 else parcel_label
            rows.append({
                "parcel_id": parcel_id,
                "parcel_label": parcel_label,
                "hemisphere": hemisphere,
                "schaefer_network": schaefer_network,
                "network_family": NETWORK_FAMILY_MAP.get(schaefer_network, schaefer_network),
                "parcel_short_name": parcel_short_name,
            })
    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError(f"No Schaefer labels could be parsed from: {label_txt_path}")
    return out

label_df = parse_schaefer_label_txt(LABEL_TXT_PATH)
label_df = label_df[label_df["schaefer_network"].isin(NETWORK_NAMES)].copy()
label_df = label_df.sort_values("parcel_id").reset_index(drop=True)

if label_df["parcel_id"].nunique() != len(label_df):
    raise ValueError("Duplicate parcel IDs detected in Schaefer label table.")

print(f"Loaded Schaefer parcel labels: {len(label_df)} parcels in target networks")
print(label_df.groupby(["network_family", "schaefer_network"]).size().reset_index(name="n_parcels").to_string(index=False))

label_lookup = label_df.set_index("parcel_id").to_dict(orient="index")
network_to_parcels = {
    network: set(label_df.loc[label_df["schaefer_network"] == network, "parcel_id"].astype(int).tolist())
    for network in NETWORK_NAMES
}

# =========================
# 3. Resample full atlas to Step 10 map grid
# =========================

reference_map = STEP10_NII_DIR / NETWORK_NAMES[0] / f"{NETWORK_NAMES[0]}_layer_00_srm_group_encoding_r.nii.gz"
if not reference_map.exists():
    raise FileNotFoundError(f"Missing reference Step 10 map: {reference_map}")

ref_nii = nib.load(str(reference_map))
atlas_nii = nib.load(str(ATLAS_PATH))
atlas_resampled_nii = image.resample_to_img(
    atlas_nii,
    ref_nii,
    interpolation="nearest",
    force_resample=True,
    copy_header=True,
)
atlas_data = np.rint(atlas_resampled_nii.get_fdata()).astype(np.int32)

print(f"Reference map shape: {ref_nii.shape}")
print(f"Resampled atlas shape: {atlas_data.shape}")

if atlas_data.shape != ref_nii.shape:
    raise ValueError(f"Atlas/reference shape mismatch: {atlas_data.shape} vs {ref_nii.shape}")

# =========================
# 4. Helper functions
# =========================

def layer_label(layer_index: int) -> str:
    if int(layer_index) == 0:
        return "embedding"
    return f"layer_{int(layer_index):02d}"


def layer_type(layer_index: int) -> str:
    return "embedding" if int(layer_index) == 0 else "transformer"


def log_layer_depth_percent(layer_index: int, n_transformer_layers: int = 48) -> float:
    if int(layer_index) <= 0:
        return 0.0
    return float(np.log1p(layer_index) / np.log1p(n_transformer_layers) * 100.0)


def layer_depth_bin(layer_index: int) -> str:
    # Log-depth bins used only for interpretation labels; exact preferred layer
    # values remain continuous through the layer_index and log-depth columns.
    if int(layer_index) == 0:
        return "embedding"
    pct = log_layer_depth_percent(layer_index)
    if pct < 50:
        return "early"
    if pct < 72:
        return "early_middle"
    if pct < 90:
        return "middle_late"
    return "late"


def finite_summary(values: np.ndarray, prefix: str) -> dict:
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {
            f"{prefix}_mean_r": np.nan,
            f"{prefix}_median_r": np.nan,
            f"{prefix}_std_r": np.nan,
            f"{prefix}_min_r": np.nan,
            f"{prefix}_max_r": np.nan,
            f"{prefix}_positive_voxel_fraction": np.nan,
        }
    return {
        f"{prefix}_mean_r": float(np.mean(values)),
        f"{prefix}_median_r": float(np.median(values)),
        f"{prefix}_std_r": float(np.std(values, ddof=0)),
        f"{prefix}_min_r": float(np.min(values)),
        f"{prefix}_max_r": float(np.max(values)),
        f"{prefix}_positive_voxel_fraction": float(np.mean(values > 0)),
    }


def centroid_from_mask(mask: np.ndarray, affine: np.ndarray):
    ijk = np.argwhere(mask)
    if ijk.size == 0:
        return (np.nan, np.nan, np.nan)
    xyz = nib.affines.apply_affine(affine, ijk)
    return tuple(float(v) for v in np.mean(xyz, axis=0))


def load_layer_maps(network: str, layer_index: int):
    layer = f"{layer_index:02d}"
    network_dir = STEP10_NII_DIR / network
    raw_path = network_dir / f"{network}_layer_{layer}_raw_group_encoding_r.nii.gz"
    srm_path = network_dir / f"{network}_layer_{layer}_srm_group_encoding_r.nii.gz"
    delta_path = network_dir / f"{network}_layer_{layer}_delta_group_encoding_r_srm_minus_raw.nii.gz"
    for path in [raw_path, srm_path, delta_path]:
        if not path.exists():
            raise FileNotFoundError(f"Missing Step 10 layer map: {path}")
    raw = np.asarray(nib.load(str(raw_path)).get_fdata(), dtype=np.float64)
    srm = np.asarray(nib.load(str(srm_path)).get_fdata(), dtype=np.float64)
    delta = np.asarray(nib.load(str(delta_path)).get_fdata(), dtype=np.float64)
    if raw.shape != atlas_data.shape or srm.shape != atlas_data.shape or delta.shape != atlas_data.shape:
        raise ValueError(f"Layer map shape mismatch for {network} layer {layer_index}")
    return raw, srm, delta, raw_path, srm_path, delta_path

# =========================
# 5. Build parcel x layer table
# =========================

rows = []
missing_networks = []
iterator = NETWORK_NAMES
if HAS_TQDM:
    iterator = tqdm(NETWORK_NAMES, desc="Networks")

for network in iterator:
    parcel_ids = sorted(network_to_parcels.get(network, []))
    if not parcel_ids:
        missing_networks.append(network)
        continue

    network_atlas_mask = np.isin(atlas_data, parcel_ids)
    if not np.any(network_atlas_mask):
        raise ValueError(f"No atlas voxels found after resampling for network: {network}")

    layer_iter = LAYER_INDICES
    if HAS_TQDM:
        layer_iter = tqdm(LAYER_INDICES, desc=f"{network} layers", leave=False)

    for li in layer_iter:
        raw_map, srm_map, delta_map, raw_path, srm_path, delta_path = load_layer_maps(network, li)

        # Retained voxels are parcel voxels with meaningful Step 10 values in at
        # least one map. This avoids summarizing atlas voxels outside the retained
        # ROI that are encoded as exact zeros in the NIfTI container.
        retained_network_mask = (
            network_atlas_mask
            & np.isfinite(raw_map)
            & np.isfinite(srm_map)
            & np.isfinite(delta_map)
            & ((np.abs(raw_map) > EPS) | (np.abs(srm_map) > EPS) | (np.abs(delta_map) > EPS))
        )

        for parcel_id in parcel_ids:
            meta = label_lookup[int(parcel_id)]
            parcel_atlas_mask = atlas_data == int(parcel_id)
            parcel_retained_mask = retained_network_mask & parcel_atlas_mask
            n_voxels_atlas = int(np.sum(parcel_atlas_mask))
            n_voxels_retained = int(np.sum(parcel_retained_mask))

            if n_voxels_retained == 0:
                # Keep an explicit row so downstream steps can diagnose parcels
                # lost after resampling/retained-voxel masking.
                cx, cy, cz = centroid_from_mask(parcel_atlas_mask, ref_nii.affine)
                row = {
                    "roi_name": network,
                    "network_family": NETWORK_FAMILY_MAP.get(network, network),
                    "parcel_id": int(parcel_id),
                    "parcel_label": meta["parcel_label"],
                    "parcel_short_name": meta["parcel_short_name"],
                    "hemisphere": meta["hemisphere"],
                    "schaefer_network": meta["schaefer_network"],
                    "layer_index": int(li),
                    "layer_label": layer_label(li),
                    "layer_type": layer_type(li),
                    "log_layer_depth_percent": log_layer_depth_percent(li),
                    "layer_depth_bin": layer_depth_bin(li),
                    "n_voxels_atlas": n_voxels_atlas,
                    "n_voxels_retained": 0,
                    "centroid_x": cx,
                    "centroid_y": cy,
                    "centroid_z": cz,
                    "raw_map_path": str(raw_path),
                    "srm_map_path": str(srm_path),
                    "delta_map_path": str(delta_path),
                }
                for prefix in ["raw", "srm", "delta"]:
                    row.update(finite_summary(np.array([], dtype=float), prefix))
                rows.append(row)
                continue

            cx, cy, cz = centroid_from_mask(parcel_retained_mask, ref_nii.affine)
            raw_values = raw_map[parcel_retained_mask]
            srm_values = srm_map[parcel_retained_mask]
            delta_values = delta_map[parcel_retained_mask]

            row = {
                "roi_name": network,
                "network_family": NETWORK_FAMILY_MAP.get(network, network),
                "parcel_id": int(parcel_id),
                "parcel_label": meta["parcel_label"],
                "parcel_short_name": meta["parcel_short_name"],
                "hemisphere": meta["hemisphere"],
                "schaefer_network": meta["schaefer_network"],
                "layer_index": int(li),
                "layer_label": layer_label(li),
                "layer_type": layer_type(li),
                "log_layer_depth_percent": log_layer_depth_percent(li),
                "layer_depth_bin": layer_depth_bin(li),
                "n_voxels_atlas": n_voxels_atlas,
                "n_voxels_retained": n_voxels_retained,
                "centroid_x": cx,
                "centroid_y": cy,
                "centroid_z": cz,
                "raw_map_path": str(raw_path),
                "srm_map_path": str(srm_path),
                "delta_map_path": str(delta_path),
            }
            row.update(finite_summary(raw_values, "raw"))
            row.update(finite_summary(srm_values, "srm"))
            row.update(finite_summary(delta_values, "delta"))
            rows.append(row)

parcel_layer_df = pd.DataFrame(rows)
if parcel_layer_df.empty:
    raise ValueError("Parcel-level encoding table is empty. Check Step 10 maps and atlas labels.")

# Stable column order for downstream cells.
front_cols = [
    "roi_name", "network_family", "parcel_id", "parcel_label", "parcel_short_name",
    "hemisphere", "schaefer_network", "layer_index", "layer_label", "layer_type",
    "log_layer_depth_percent", "layer_depth_bin", "n_voxels_atlas", "n_voxels_retained",
    "centroid_x", "centroid_y", "centroid_z",
]
metric_cols = [c for c in parcel_layer_df.columns if c not in front_cols and not c.endswith("_map_path")]
path_cols = [c for c in parcel_layer_df.columns if c.endswith("_map_path")]
parcel_layer_df = parcel_layer_df[front_cols + metric_cols + path_cols]

# =========================
# 6. QA summaries
# =========================

parcel_summary_df = (
    parcel_layer_df
    .groupby(["roi_name", "network_family", "parcel_id", "parcel_label", "hemisphere"], as_index=False)
    .agg(
        n_layers=("layer_index", "nunique"),
        n_voxels_atlas=("n_voxels_atlas", "first"),
        n_voxels_retained=("n_voxels_retained", "max"),
        centroid_x=("centroid_x", "first"),
        centroid_y=("centroid_y", "first"),
        centroid_z=("centroid_z", "first"),
        mean_raw_r_across_layers=("raw_mean_r", "mean"),
        mean_srm_r_across_layers=("srm_mean_r", "mean"),
        mean_delta_r_across_layers=("delta_mean_r", "mean"),
    )
)

network_summary_df = (
    parcel_summary_df
    .groupby(["roi_name", "network_family"], as_index=False)
    .agg(
        n_parcels=("parcel_id", "nunique"),
        total_retained_voxels=("n_voxels_retained", "sum"),
        median_retained_voxels_per_parcel=("n_voxels_retained", "median"),
        mean_raw_r_across_parcels_layers=("mean_raw_r_across_layers", "mean"),
        mean_srm_r_across_parcels_layers=("mean_srm_r_across_layers", "mean"),
        mean_delta_r_across_parcels_layers=("mean_delta_r_across_layers", "mean"),
    )
)

missing_parcel_rows = parcel_summary_df[parcel_summary_df["n_voxels_retained"] == 0].copy()

# =========================
# 7. Save tables and QA figures
# =========================

parcel_layer_csv = CSV_DIR / "step11A_parcel_layer_encoding_table.csv"
parcel_summary_csv = CSV_DIR / "step11A_parcel_summary.csv"
network_summary_csv = CSV_DIR / "step11A_network_parcel_summary.csv"
missing_parcel_csv = CSV_DIR / "step11A_missing_or_empty_parcels.csv"
label_csv = CSV_DIR / "step11A_schaefer_labels_used.csv"

parcel_layer_df.to_csv(parcel_layer_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
parcel_summary_df.to_csv(parcel_summary_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
network_summary_df.to_csv(network_summary_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
missing_parcel_rows.to_csv(missing_parcel_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
label_df.to_csv(label_csv, index=False, encoding="utf-8-sig")

try:
    xlsx_path = XLSX_DIR / "step11A_parcel_level_encoding_tables.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        parcel_layer_df.to_excel(writer, sheet_name="parcel_layer_encoding", index=False)
        parcel_summary_df.to_excel(writer, sheet_name="parcel_summary", index=False)
        network_summary_df.to_excel(writer, sheet_name="network_summary", index=False)
        missing_parcel_rows.to_excel(writer, sheet_name="empty_parcels", index=False)
        label_df.to_excel(writer, sheet_name="schaefer_labels_used", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")

sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(figsize=(12, 6))
plot_df = network_summary_df.set_index("roi_name").loc[NETWORK_NAMES].reset_index()
sns.barplot(data=plot_df, x="roi_name", y="n_parcels", color="#4C72B0", ax=ax)
ax.set_title("Schaefer parcels per network")
ax.set_xlabel("Network")
ax.set_ylabel("Number of parcels")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(FIG_DIR / "step11A_parcels_per_network.png", dpi=260, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(data=parcel_summary_df, x="n_voxels_retained", bins=40, color="#55A868", ax=ax)
ax.set_title("Retained voxels per Schaefer parcel")
ax.set_xlabel("Retained voxels")
ax.set_ylabel("Number of parcels")
plt.tight_layout()
fig.savefig(FIG_DIR / "step11A_retained_voxels_per_parcel_histogram.png", dpi=260, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
metric_plot_df = network_summary_df.melt(
    id_vars=["roi_name", "network_family"],
    value_vars=["mean_raw_r_across_parcels_layers", "mean_srm_r_across_parcels_layers", "mean_delta_r_across_parcels_layers"],
    var_name="metric",
    value_name="mean_r",
)
metric_plot_df["metric"] = metric_plot_df["metric"].replace({
    "mean_raw_r_across_parcels_layers": "Raw + LLM",
    "mean_srm_r_across_parcels_layers": "SRM + LLM",
    "mean_delta_r_across_parcels_layers": "SRM - Raw",
})
sns.barplot(data=metric_plot_df, x="roi_name", y="mean_r", hue="metric", ax=ax)
ax.set_title("Parcel-level encoding summaries by network")
ax.set_xlabel("Network")
ax.set_ylabel("Mean encoding r")
ax.tick_params(axis="x", rotation=45)
ax.legend(title=None, frameon=False)
plt.tight_layout()
fig.savefig(FIG_DIR / "step11A_network_mean_encoding_summary.png", dpi=260, bbox_inches="tight")
plt.close(fig)

summary = {
    "step": "step11A_build_parcel_level_encoding_table",
    "description": "Summarize Step 10 voxel-wise layer encoding maps into Schaefer parcel-level tables.",
    "n_networks_requested": int(len(NETWORK_NAMES)),
    "n_layers": int(len(LAYER_INDICES)),
    "n_parcels_in_labels": int(label_df["parcel_id"].nunique()),
    "n_parcel_layer_rows": int(len(parcel_layer_df)),
    "n_empty_parcels": int(len(missing_parcel_rows)),
    "atlas_path": str(ATLAS_PATH),
    "label_txt_path": str(LABEL_TXT_PATH),
    "reference_map": str(reference_map),
    "outputs": {
        "parcel_layer_table_csv": str(parcel_layer_csv),
        "parcel_summary_csv": str(parcel_summary_csv),
        "network_summary_csv": str(network_summary_csv),
        "missing_parcel_csv": str(missing_parcel_csv),
        "label_csv": str(label_csv),
        "figure_dir": str(FIG_DIR),
    },
    "elapsed_minutes": float((time.time() - start_time) / 60.0),
}

with open(JSON_DIR / "step11A_parcel_level_encoding_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11A completed: parcel-level encoding table built")
print(f"Parcel-layer rows : {len(parcel_layer_df)}")
print(f"Parcel summary rows: {len(parcel_summary_df)}")
print(f"Empty parcels     : {len(missing_parcel_rows)}")
print(f"Saved main table  : {parcel_layer_csv}")
print(f"Saved summary JSON: {JSON_DIR / 'step11A_parcel_level_encoding_summary.json'}")
print("\nNetwork summary:")
display(network_summary_df)
print("\nFirst parcel-layer rows:")
display(parcel_layer_df.head())


In [ ]:
# Step 11B: Compute Parcel-Level Preferred Layer And Reliability Metrics
#
# This cell summarizes the parcel x layer table from Step 11A into one row per
# Schaefer parcel. It keeps the intuitive argmax preferred layer, but also adds
# a weighted depth center of mass and reliability metrics so weak/noisy parcel
# peaks are not over-interpreted.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

pd.options.display.float_format = "{:.6f}".format
np.set_printoptions(suppress=True, precision=6)

# =========================
# 1. Project setup
# =========================

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
STEP11A_DIR = BASE_DIR / "parcel_layer_network_outputs" / "step11A_build_parcel_level_encoding_table"
STEP11B_DIR = BASE_DIR / "parcel_layer_network_outputs" / "step11B_parcel_preferred_layer"

CSV_DIR = STEP11B_DIR / "csv"
XLSX_DIR = STEP11B_DIR / "xlsx"
FIG_DIR = STEP11B_DIR / "figures"
JSON_DIR = STEP11B_DIR / "json"
for d in [STEP11B_DIR, CSV_DIR, XLSX_DIR, FIG_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PARCEL_LAYER_TABLE_PATH = STEP11A_DIR / "csv" / "step11A_parcel_layer_encoding_table.csv"
if not PARCEL_LAYER_TABLE_PATH.exists():
    raise FileNotFoundError(f"Missing Step 11A parcel-layer table: {PARCEL_LAYER_TABLE_PATH}")

# Main preferred-layer estimates are based on transformer layers only. The
# embedding row is retained in Step 11A but excluded here because layer-depth
# interpretation refers to GPT-2 XL transformer layers 1-48.
USE_TRANSFORMER_LAYERS_ONLY = True
TOP_K_LAYERS = 3
MIN_RETAINED_VOXELS = 20
MIN_BEST_SRM_R = 0.020
MIN_BEST_MARGIN = 0.005
MIN_DYNAMIC_RANGE = 0.010
EPS = 1e-8

start_time = time.time()
print("Step 11B: parcel-level preferred layer and reliability metrics")
print(f"Input table : {PARCEL_LAYER_TABLE_PATH}")
print(f"Output dir  : {STEP11B_DIR}")
print(f"Transformer-only preferred layer: {USE_TRANSFORMER_LAYERS_ONLY}")

# =========================
# 2. Load and validate Step 11A table
# =========================

parcel_layer_df = pd.read_csv(PARCEL_LAYER_TABLE_PATH)
required_cols = [
    "roi_name", "network_family", "parcel_id", "parcel_label", "parcel_short_name",
    "hemisphere", "schaefer_network", "layer_index", "layer_label", "layer_type",
    "log_layer_depth_percent", "layer_depth_bin", "n_voxels_atlas", "n_voxels_retained",
    "centroid_x", "centroid_y", "centroid_z",
    "raw_mean_r", "srm_mean_r", "delta_mean_r",
    "raw_median_r", "srm_median_r", "delta_median_r",
]
missing_cols = [c for c in required_cols if c not in parcel_layer_df.columns]
if missing_cols:
    raise ValueError(f"Step 11A table is missing required columns: {missing_cols}")

parcel_layer_df = parcel_layer_df.replace([np.inf, -np.inf], np.nan)
analysis_df = parcel_layer_df.copy()
if USE_TRANSFORMER_LAYERS_ONLY:
    analysis_df = analysis_df[analysis_df["layer_index"] > 0].copy()

if analysis_df.empty:
    raise ValueError("No rows available for preferred-layer analysis after layer filtering.")

print(f"Step 11A rows loaded : {len(parcel_layer_df)}")
print(f"Rows used in Step11B : {len(analysis_df)}")
print(f"Unique parcels       : {analysis_df['parcel_id'].nunique()}")
print(f"Layer range used     : {int(analysis_df['layer_index'].min())} to {int(analysis_df['layer_index'].max())}")

# =========================
# 3. Helper functions
# =========================

def safe_weighted_center(depth_values, metric_values, positive_only=True):
    depth_values = np.asarray(depth_values, dtype=np.float64)
    metric_values = np.asarray(metric_values, dtype=np.float64)
    valid = np.isfinite(depth_values) & np.isfinite(metric_values)
    if not np.any(valid):
        return np.nan
    depth_values = depth_values[valid]
    metric_values = metric_values[valid]
    weights = np.clip(metric_values, 0.0, None) if positive_only else np.abs(metric_values)
    if np.sum(weights) <= EPS:
        return np.nan
    return float(np.sum(depth_values * weights) / np.sum(weights))


def top_k_summary(sub, metric_col, top_k=TOP_K_LAYERS):
    valid = sub[np.isfinite(sub[metric_col])].copy()
    if valid.empty:
        return {
            f"top{top_k}_{metric_col}_layers": "",
            f"top{top_k}_{metric_col}_mean_layer": np.nan,
            f"top{top_k}_{metric_col}_mean_log_depth_percent": np.nan,
            f"top{top_k}_{metric_col}_mean_r": np.nan,
        }
    top = valid.sort_values(metric_col, ascending=False).head(top_k)
    return {
        f"top{top_k}_{metric_col}_layers": ",".join(top["layer_label"].astype(str).tolist()),
        f"top{top_k}_{metric_col}_mean_layer": float(top["layer_index"].mean()),
        f"top{top_k}_{metric_col}_mean_log_depth_percent": float(top["log_layer_depth_percent"].mean()),
        f"top{top_k}_{metric_col}_mean_r": float(top[metric_col].mean()),
    }


def best_layer_summary(sub, metric_col, prefix):
    valid = sub[np.isfinite(sub[metric_col])].copy()
    if valid.empty:
        return {
            f"best_{prefix}_layer_index": np.nan,
            f"best_{prefix}_layer_label": None,
            f"best_{prefix}_log_depth_percent": np.nan,
            f"best_{prefix}_depth_bin": None,
            f"best_{prefix}_r": np.nan,
            f"{prefix}_second_best_r": np.nan,
            f"{prefix}_best_minus_second": np.nan,
            f"{prefix}_best_minus_median": np.nan,
            f"{prefix}_dynamic_range": np.nan,
            f"{prefix}_positive_layer_fraction": np.nan,
            f"{prefix}_depth_center_of_mass": np.nan,
        }

    sorted_valid = valid.sort_values(metric_col, ascending=False)
    best = sorted_valid.iloc[0]
    second_best = sorted_valid.iloc[1][metric_col] if len(sorted_valid) > 1 else np.nan
    metric_values = valid[metric_col].astype(float).values
    center = safe_weighted_center(
        valid["log_layer_depth_percent"].values,
        metric_values,
        positive_only=True,
    )

    return {
        f"best_{prefix}_layer_index": int(best["layer_index"]),
        f"best_{prefix}_layer_label": best["layer_label"],
        f"best_{prefix}_log_depth_percent": float(best["log_layer_depth_percent"]),
        f"best_{prefix}_depth_bin": best["layer_depth_bin"],
        f"best_{prefix}_r": float(best[metric_col]),
        f"{prefix}_second_best_r": float(second_best) if np.isfinite(second_best) else np.nan,
        f"{prefix}_best_minus_second": float(best[metric_col] - second_best) if np.isfinite(second_best) else np.nan,
        f"{prefix}_best_minus_median": float(best[metric_col] - np.nanmedian(metric_values)),
        f"{prefix}_dynamic_range": float(np.nanmax(metric_values) - np.nanmin(metric_values)),
        f"{prefix}_positive_layer_fraction": float(np.mean(metric_values > 0)),
        f"{prefix}_depth_center_of_mass": center,
    }


def confidence_tier(best_r, margin, dynamic_range, n_voxels):
    if not np.isfinite(best_r) or not np.isfinite(margin) or not np.isfinite(dynamic_range):
        return "low"
    passes_voxel = n_voxels >= MIN_RETAINED_VOXELS
    passes_best = best_r >= MIN_BEST_SRM_R
    passes_margin = margin >= MIN_BEST_MARGIN
    passes_range = dynamic_range >= MIN_DYNAMIC_RANGE
    n_pass = int(passes_voxel) + int(passes_best) + int(passes_margin) + int(passes_range)
    if n_pass == 4:
        return "high"
    if n_pass >= 2:
        return "moderate"
    return "low"

# =========================
# 4. Compute parcel-level preferred-layer summaries
# =========================

id_cols = [
    "roi_name", "network_family", "parcel_id", "parcel_label", "parcel_short_name",
    "hemisphere", "schaefer_network", "n_voxels_atlas", "n_voxels_retained",
    "centroid_x", "centroid_y", "centroid_z",
]

rows = []
for keys, sub in analysis_df.groupby(id_cols, dropna=False):
    row = dict(zip(id_cols, keys))
    sub = sub.sort_values("layer_index").copy()

    row.update(best_layer_summary(sub, "raw_mean_r", "raw"))
    row.update(best_layer_summary(sub, "srm_mean_r", "srm"))
    row.update(best_layer_summary(sub, "delta_mean_r", "delta"))

    row.update(top_k_summary(sub, "srm_mean_r", TOP_K_LAYERS))
    row.update(top_k_summary(sub, "delta_mean_r", TOP_K_LAYERS))

    # Same-layer gain: compare SRM and Raw at the SRM-preferred layer. This keeps
    # the SRM-gain interpretation aligned with the layer that best predicts the
    # SRM-reconstructed response.
    best_srm_layer = row.get("best_srm_layer_index")
    if np.isfinite(best_srm_layer):
        best_layer_row = sub[sub["layer_index"] == int(best_srm_layer)].iloc[0]
        row["raw_r_at_best_srm_layer"] = float(best_layer_row["raw_mean_r"])
        row["srm_r_at_best_srm_layer"] = float(best_layer_row["srm_mean_r"])
        row["same_layer_srm_gain_at_best_srm_layer"] = float(best_layer_row["srm_mean_r"] - best_layer_row["raw_mean_r"])
    else:
        row["raw_r_at_best_srm_layer"] = np.nan
        row["srm_r_at_best_srm_layer"] = np.nan
        row["same_layer_srm_gain_at_best_srm_layer"] = np.nan

    # A difference between argmax depth and center-of-mass depth is useful as a
    # noise/dispersion warning: a sharp peak and a broad positive plateau can have
    # the same best layer but different centers of mass.
    row["srm_best_vs_center_depth_difference"] = (
        row["best_srm_log_depth_percent"] - row["srm_depth_center_of_mass"]
        if np.isfinite(row["best_srm_log_depth_percent"]) and np.isfinite(row["srm_depth_center_of_mass"])
        else np.nan
    )

    row["srm_preferred_layer_confidence"] = confidence_tier(
        row["best_srm_r"],
        row["srm_best_minus_second"],
        row["srm_dynamic_range"],
        row["n_voxels_retained"],
    )
    row["srm_preferred_layer_is_interpretable"] = bool(row["srm_preferred_layer_confidence"] in ["high", "moderate"])

    rows.append(row)

preferred_df = pd.DataFrame(rows)
preferred_df = preferred_df.sort_values(["roi_name", "hemisphere", "parcel_id"]).reset_index(drop=True)

if preferred_df.empty:
    raise ValueError("Preferred-layer table is empty.")

# =========================
# 5. Network-level summaries
# =========================

confidence_order = pd.CategoricalDtype(["low", "moderate", "high"], ordered=True)
preferred_df["srm_preferred_layer_confidence"] = preferred_df["srm_preferred_layer_confidence"].astype(confidence_order)

network_preferred_summary_df = (
    preferred_df
    .groupby(["roi_name", "network_family"], as_index=False)
    .agg(
        n_parcels=("parcel_id", "nunique"),
        n_interpretable_parcels=("srm_preferred_layer_is_interpretable", "sum"),
        mean_best_srm_r=("best_srm_r", "mean"),
        median_best_srm_r=("best_srm_r", "median"),
        mean_same_layer_srm_gain=("same_layer_srm_gain_at_best_srm_layer", "mean"),
        mean_best_srm_layer=("best_srm_layer_index", "mean"),
        median_best_srm_layer=("best_srm_layer_index", "median"),
        mean_best_srm_log_depth_percent=("best_srm_log_depth_percent", "mean"),
        median_best_srm_log_depth_percent=("best_srm_log_depth_percent", "median"),
        mean_srm_depth_center_of_mass=("srm_depth_center_of_mass", "mean"),
        sd_srm_depth_center_of_mass=("srm_depth_center_of_mass", "std"),
        mean_srm_best_minus_second=("srm_best_minus_second", "mean"),
        mean_srm_dynamic_range=("srm_dynamic_range", "mean"),
        mean_retained_voxels=("n_voxels_retained", "mean"),
    )
)
network_preferred_summary_df["interpretable_parcel_fraction"] = (
    network_preferred_summary_df["n_interpretable_parcels"] / network_preferred_summary_df["n_parcels"]
)

bin_counts_df = (
    preferred_df
    .pivot_table(
        index=["roi_name", "network_family"],
        columns="best_srm_depth_bin",
        values="parcel_id",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)
for col in ["early", "early_middle", "middle_late", "late"]:
    if col not in bin_counts_df.columns:
        bin_counts_df[col] = 0
bin_counts_df["n_parcels"] = bin_counts_df[["early", "early_middle", "middle_late", "late"]].sum(axis=1)
for col in ["early", "early_middle", "middle_late", "late"]:
    bin_counts_df[f"{col}_parcel_fraction"] = bin_counts_df[col] / bin_counts_df["n_parcels"]

confidence_counts_df = (
    preferred_df
    .pivot_table(
        index=["roi_name", "network_family"],
        columns="srm_preferred_layer_confidence",
        values="parcel_id",
        aggfunc="count",
        fill_value=0,
        observed=True,
    )
    .reset_index()
)
for col in ["low", "moderate", "high"]:
    if col not in confidence_counts_df.columns:
        confidence_counts_df[col] = 0
confidence_counts_df["n_parcels"] = confidence_counts_df[["low", "moderate", "high"]].sum(axis=1)
confidence_counts_df = confidence_counts_df[confidence_counts_df["n_parcels"] > 0].copy()
for col in ["low", "moderate", "high"]:
    confidence_counts_df[f"{col}_confidence_fraction"] = confidence_counts_df[col] / confidence_counts_df["n_parcels"]

# =========================
# 6. Save tables
# =========================

preferred_csv = CSV_DIR / "step11B_parcel_preferred_layer_table.csv"
network_summary_csv = CSV_DIR / "step11B_network_preferred_layer_summary.csv"
bin_counts_csv = CSV_DIR / "step11B_network_preferred_layer_bin_counts.csv"
confidence_counts_csv = CSV_DIR / "step11B_network_confidence_counts.csv"

top_srm_csv = CSV_DIR / "step11B_top_parcels_by_best_srm_r.csv"
top_gain_csv = CSV_DIR / "step11B_top_parcels_by_same_layer_srm_gain.csv"

preferred_df.to_csv(preferred_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
network_preferred_summary_df.to_csv(network_summary_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
bin_counts_df.to_csv(bin_counts_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
confidence_counts_df.to_csv(confidence_counts_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
preferred_df.sort_values("best_srm_r", ascending=False).head(50).to_csv(top_srm_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
preferred_df.sort_values("same_layer_srm_gain_at_best_srm_layer", ascending=False).head(50).to_csv(top_gain_csv, index=False, encoding="utf-8-sig", float_format="%.6f")

try:
    xlsx_path = XLSX_DIR / "step11B_parcel_preferred_layer_tables.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        preferred_df.to_excel(writer, sheet_name="parcel_preferred_layer", index=False)
        network_preferred_summary_df.to_excel(writer, sheet_name="network_summary", index=False)
        bin_counts_df.to_excel(writer, sheet_name="depth_bin_counts", index=False)
        confidence_counts_df.to_excel(writer, sheet_name="confidence_counts", index=False)
        preferred_df.sort_values("best_srm_r", ascending=False).head(50).to_excel(writer, sheet_name="top_best_srm", index=False)
        preferred_df.sort_values("same_layer_srm_gain_at_best_srm_layer", ascending=False).head(50).to_excel(writer, sheet_name="top_srm_gain", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")

# =========================
# 7. QA figures
# =========================

sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(
    data=preferred_df,
    x="best_srm_log_depth_percent",
    hue="network_family",
    bins=24,
    multiple="stack",
    ax=ax,
)
ax.set_title("Parcel-level SRM+LLM preferred layer depth")
ax.set_xlabel("Best Encoding Layer (%)")
ax.set_ylabel("Number of parcels")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
leg = ax.get_legend()
if leg is not None:
    leg.set_title("Network family")
    leg.set_bbox_to_anchor((1.02, 1.0))
    leg._loc = 2
    leg.set_frame_on(False)
plt.tight_layout(rect=[0, 0, 0.82, 1])
fig.savefig(FIG_DIR / "step11B_parcel_best_srm_layer_depth_histogram.png", dpi=260, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(13, 6))
plot_df = network_preferred_summary_df.sort_values("mean_srm_depth_center_of_mass", ascending=False)
sns.barplot(data=plot_df, x="roi_name", y="mean_srm_depth_center_of_mass", color="#C44E52", ax=ax)
ax.set_title("Mean parcel-level SRM+LLM depth center of mass by network")
ax.set_xlabel("Network")
ax.set_ylabel("Depth center of mass (%)")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))
plt.tight_layout()
fig.savefig(FIG_DIR / "step11B_network_mean_depth_center_of_mass.png", dpi=260, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(13, 6))
plot_df = network_preferred_summary_df.sort_values("mean_same_layer_srm_gain", ascending=False)
sns.barplot(data=plot_df, x="roi_name", y="mean_same_layer_srm_gain", color="#55A868", ax=ax)
ax.set_title("Mean parcel-level SRM gain at each parcel's SRM-preferred layer")
ax.set_xlabel("Network")
ax.set_ylabel("SRM - Raw encoding r")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
plt.tight_layout()
fig.savefig(FIG_DIR / "step11B_network_mean_same_layer_srm_gain.png", dpi=260, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(13, 6))
conf_plot_df = confidence_counts_df.melt(
    id_vars=["roi_name", "network_family"],
    value_vars=["low_confidence_fraction", "moderate_confidence_fraction", "high_confidence_fraction"],
    var_name="confidence",
    value_name="fraction",
)
conf_plot_df["confidence"] = conf_plot_df["confidence"].str.replace("_confidence_fraction", "", regex=False)
sns.barplot(data=conf_plot_df, x="roi_name", y="fraction", hue="confidence", ax=ax)
ax.set_title("Preferred-layer confidence by network")
ax.set_xlabel("Network")
ax.set_ylabel("Parcel fraction")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Confidence", frameon=False, bbox_to_anchor=(1.02, 1.0), loc="upper left")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
plt.tight_layout(rect=[0, 0, 0.82, 1])
fig.savefig(FIG_DIR / "step11B_network_preferred_layer_confidence.png", dpi=260, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=preferred_df,
    x="best_srm_log_depth_percent",
    y="srm_depth_center_of_mass",
    hue="network_family",
    size="best_srm_r",
    sizes=(20, 160),
    alpha=0.75,
    ax=ax,
)
ax.plot([0, 100], [0, 100], color="black", linestyle="--", linewidth=1)
ax.set_title("Argmax depth vs weighted depth center of mass")
ax.set_xlabel("Argmax Best Encoding Layer (%)")
ax.set_ylabel("Depth center of mass (%)")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))
ax.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left", frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0, 0.78, 1])
fig.savefig(FIG_DIR / "step11B_argmax_depth_vs_center_of_mass.png", dpi=260, bbox_inches="tight")
plt.close(fig)

# =========================
# 8. Summary JSON and display
# =========================

summary = {
    "step": "step11B_parcel_preferred_layer",
    "description": "Compute parcel-level preferred GPT-2 layer, depth center of mass, and reliability/confidence metrics.",
    "input_table": str(PARCEL_LAYER_TABLE_PATH),
    "use_transformer_layers_only": bool(USE_TRANSFORMER_LAYERS_ONLY),
    "n_parcels": int(preferred_df["parcel_id"].nunique()),
    "n_networks": int(preferred_df["roi_name"].nunique()),
    "top_k_layers": int(TOP_K_LAYERS),
    "confidence_thresholds": {
        "min_retained_voxels": int(MIN_RETAINED_VOXELS),
        "min_best_srm_r": float(MIN_BEST_SRM_R),
        "min_best_margin": float(MIN_BEST_MARGIN),
        "min_dynamic_range": float(MIN_DYNAMIC_RANGE),
    },
    "confidence_counts": preferred_df["srm_preferred_layer_confidence"].astype(str).value_counts().to_dict(),
    "mean_best_srm_r": float(preferred_df["best_srm_r"].mean()),
    "mean_same_layer_srm_gain": float(preferred_df["same_layer_srm_gain_at_best_srm_layer"].mean()),
    "mean_srm_depth_center_of_mass": float(preferred_df["srm_depth_center_of_mass"].mean()),
    "outputs": {
        "preferred_layer_table_csv": str(preferred_csv),
        "network_summary_csv": str(network_summary_csv),
        "bin_counts_csv": str(bin_counts_csv),
        "confidence_counts_csv": str(confidence_counts_csv),
        "top_srm_csv": str(top_srm_csv),
        "top_gain_csv": str(top_gain_csv),
        "figure_dir": str(FIG_DIR),
    },
    "elapsed_minutes": float((time.time() - start_time) / 60.0),
}

with open(JSON_DIR / "step11B_parcel_preferred_layer_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11B completed: parcel-level preferred layer table built")
print(f"Saved preferred layer table: {preferred_csv}")
print(f"Saved network summary      : {network_summary_csv}")
print("\nConfidence counts:")
print(preferred_df["srm_preferred_layer_confidence"].astype(str).value_counts().to_string())
print("\nTop parcels by best SRM+LLM r:")
display(preferred_df.sort_values("best_srm_r", ascending=False).head(20))
print("\nNetwork preferred-layer summary:")
display(network_preferred_summary_df.sort_values("mean_best_srm_r", ascending=False))


In [ ]:
# Step 11C: Parcel-Level Preferred Layer Brain Maps
#
# This cell maps the parcel-level preferred-layer results from Step 11B back
# into brain space. It creates NIfTI maps and summary figures for SRM+LLM
# preferred layer depth, depth center of mass, best SRM+LLM encoding strength,
# and SRM-related gain at each parcel's preferred layer.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from nilearn import image
from nilearn.plotting import plot_glass_brain, plot_stat_map

pd.options.display.float_format = "{:.6f}".format
np.set_printoptions(suppress=True, precision=6)

# =========================
# 1. Project setup
# =========================

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
PROJECT_DIR = BASE_DIR / "SNL_layerwise_encoding_fixed_srm50"
STEP10_DIR = PROJECT_DIR / "step10_layerwise_brain_mapping"
STEP10_NII_DIR = STEP10_DIR / "nii"
STEP11B_DIR = BASE_DIR / "parcel_layer_network_outputs" / "step11B_parcel_preferred_layer"
STEP11C_DIR = BASE_DIR / "parcel_layer_network_outputs" / "step11C_parcel_preferred_layer_brain_maps"

NII_DIR = STEP11C_DIR / "nii"
CSV_DIR = STEP11C_DIR / "csv"
JSON_DIR = STEP11C_DIR / "json"
FIG_GLASS_DIR = STEP11C_DIR / "figures_glass"
FIG_ORTHO_DIR = STEP11C_DIR / "figures_ortho_summary"
FIG_XYZ_DIR = STEP11C_DIR / "figures_xyz_directional_slices"
for d in [STEP11C_DIR, NII_DIR, CSV_DIR, JSON_DIR, FIG_GLASS_DIR, FIG_ORTHO_DIR, FIG_XYZ_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PREFERRED_TABLE_PATH = STEP11B_DIR / "csv" / "step11B_parcel_preferred_layer_table.csv"
ATLAS_PATH = Path(r"YOUR_SCHAEFER400_ATLAS_NIFTI_PATH")
REFERENCE_MAP = STEP10_NII_DIR / "VisCent" / "VisCent_layer_00_srm_group_encoding_r.nii.gz"

for p in [PREFERRED_TABLE_PATH, ATLAS_PATH, REFERENCE_MAP]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing required input: {p}")

start_time = time.time()
print("Step 11C: parcel-level preferred-layer brain maps")
print(f"Preferred table: {PREFERRED_TABLE_PATH}")
print(f"Output dir     : {STEP11C_DIR}")

# =========================
# 2. Load preferred-layer table and atlas
# =========================

preferred_df = pd.read_csv(PREFERRED_TABLE_PATH).replace([np.inf, -np.inf], np.nan)
required_cols = [
    "parcel_id", "parcel_label", "roi_name", "network_family", "hemisphere",
    "best_srm_log_depth_percent", "srm_depth_center_of_mass", "best_srm_r",
    "same_layer_srm_gain_at_best_srm_layer", "srm_preferred_layer_confidence",
    "srm_preferred_layer_is_interpretable",
]
missing_cols = [c for c in required_cols if c not in preferred_df.columns]
if missing_cols:
    raise ValueError(f"Step 11B preferred-layer table is missing required columns: {missing_cols}")

# Normalize boolean column if it was read from CSV as text.
if preferred_df["srm_preferred_layer_is_interpretable"].dtype == object:
    preferred_df["srm_preferred_layer_is_interpretable"] = preferred_df["srm_preferred_layer_is_interpretable"].astype(str).str.lower().isin(["true", "1", "yes"])

ref_nii = nib.load(str(REFERENCE_MAP))
atlas_nii = nib.load(str(ATLAS_PATH))
atlas_resampled_nii = image.resample_to_img(
    atlas_nii,
    ref_nii,
    interpolation="nearest",
    force_resample=True,
    copy_header=True,
)
atlas_data = np.rint(atlas_resampled_nii.get_fdata()).astype(np.int32)

if atlas_data.shape != ref_nii.shape:
    raise ValueError(f"Atlas/reference shape mismatch: {atlas_data.shape} vs {ref_nii.shape}")

print(f"Preferred parcels: {preferred_df['parcel_id'].nunique()}")
print(f"Reference shape  : {ref_nii.shape}")

# =========================
# 3. Helper functions
# =========================

def parcel_values_to_nifti(value_map, out_path, background_value=0.0):
    out = np.full(atlas_data.shape, background_value, dtype=np.float32)
    for parcel_id, value in value_map.items():
        if not np.isfinite(value):
            continue
        out[atlas_data == int(parcel_id)] = np.float32(value)
    nii = nib.Nifti1Image(out, ref_nii.affine, ref_nii.header)
    nib.save(nii, str(out_path))
    return out_path


def format_colorbar(disp, kind="decimal", label=None):
    try:
        cbar = disp._cbar
        if kind == "depth":
            cbar.set_ticks([0, 25, 50, 75, 100])
            fmt = "%.0f"
        elif kind == "r":
            fmt = "%.3f"
        else:
            fmt = "%.5f"

        formatter = FormatStrFormatter(fmt)
        for axis in [cbar.ax.yaxis, cbar.ax.xaxis]:
            axis.set_major_formatter(formatter)
            axis.get_offset_text().set_visible(False)
            try:
                axis.major.formatter.set_scientific(False)
                axis.major.formatter.set_useOffset(False)
            except Exception:
                pass

        # Matplotlib/nilearn can create scientific-notation offset text after
        # update_ticks(), so hide it both before and after the update.
        cbar.update_ticks()
        cbar.ax.yaxis.get_offset_text().set_visible(False)
        cbar.ax.xaxis.get_offset_text().set_visible(False)
        cbar.ax.yaxis.offsetText.set_visible(False)
        cbar.ax.xaxis.offsetText.set_visible(False)

        if label:
            cbar.ax.set_ylabel(label, rotation=90, labelpad=10)
    except Exception:
        pass


def safe_vmax(values, percentile=99.0, floor=0.00001):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return floor
    vmax = float(np.nanpercentile(values, percentile))
    if not np.isfinite(vmax) or vmax <= 0:
        return floor
    return vmax


def safe_absmax(values, percentile=99.0, floor=0.00001):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return floor
    vmax = float(np.nanpercentile(np.abs(values), percentile))
    if not np.isfinite(vmax) or vmax <= 0:
        return floor
    return vmax


def save_glass_map(img_path, out_path, title, cmap, vmin, vmax, threshold, cbar_kind, cbar_label, symmetric_cbar=False):
    disp = plot_glass_brain(
        str(img_path),
        display_mode="lyrz",
        colorbar=True,
        cmap=cmap,
        plot_abs=False,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        symmetric_cbar=symmetric_cbar,
        title=title,
        cbar_tick_format="%.0f" if cbar_kind == "depth" else ("%.3f" if cbar_kind == "r" else "%.5f"),
    )
    format_colorbar(disp, kind=cbar_kind, label=cbar_label)
    disp.savefig(out_path, dpi=240, bbox_inches="tight", pad_inches=0.08)
    plt.close()


def save_ortho_map(img_path, out_path, title, cmap, vmin, vmax, threshold, cbar_kind, cbar_label, symmetric_cbar=False):
    disp = plot_stat_map(
        str(img_path),
        display_mode="ortho",
        cut_coords=(0, 0, 0),
        colorbar=True,
        cmap=cmap,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        black_bg=False,
        dim=-0.4,
        draw_cross=True,
        symmetric_cbar=symmetric_cbar,
        title=title,
        cbar_tick_format="%.0f" if cbar_kind == "depth" else ("%.3f" if cbar_kind == "r" else "%.5f"),
    )
    format_colorbar(disp, kind=cbar_kind, label=cbar_label)
    disp.savefig(out_path, dpi=260, bbox_inches="tight", pad_inches=0.08)
    plt.close()


def short_map_id(map_id):
    short_lookup = {
        "parcel_best_srm_log_layer_depth_percent": "best_depth",
        "parcel_srm_depth_center_of_mass_percent": "depth_com",
        "parcel_best_srm_group_encoding_r": "best_srm_r",
        "parcel_same_layer_srm_gain_at_best_srm_layer": "srm_gain",
        "parcel_preferred_layer_confidence": "confidence",
        "parcel_best_srm_log_layer_depth_percent_interpretable_only": "best_depth_interp",
    }
    return short_lookup.get(map_id, map_id[:40])


def save_directional_map(img_path, out_path, title, display_mode, cmap, vmin, vmax, threshold, cbar_kind, cbar_label, symmetric_cbar=False):
    disp = plot_stat_map(
        str(img_path),
        display_mode=display_mode,
        cut_coords=7,
        colorbar=True,
        cmap=cmap,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        black_bg=False,
        symmetric_cbar=symmetric_cbar,
        title=title,
        cbar_tick_format="%.0f" if cbar_kind == "depth" else ("%.3f" if cbar_kind == "r" else "%.5f"),
    )
    format_colorbar(disp, kind=cbar_kind, label=cbar_label)
    disp.savefig(out_path, dpi=260, bbox_inches="tight", pad_inches=0.08)
    plt.close()

# =========================
# 4. Build parcel-value maps
# =========================

preferred_df["confidence_numeric"] = preferred_df["srm_preferred_layer_confidence"].map({"low": 1, "moderate": 2, "high": 3}).astype(float)

map_specs = [
    {
        "map_id": "parcel_best_srm_log_layer_depth_percent",
        "column": "best_srm_log_depth_percent",
        "title": "Parcel preferred GPT-2 layer depth",
        "cmap": "inferno",
        "vmin": 0.0,
        "vmax": 100.0,
        "threshold": 0.001,
        "cbar_kind": "depth",
        "cbar_label": "Best Encoding Layer (%)",
        "symmetric": False,
        "description": "Argmax SRM+LLM preferred layer depth for each Schaefer parcel.",
    },
    {
        "map_id": "parcel_srm_depth_center_of_mass_percent",
        "column": "srm_depth_center_of_mass",
        "title": "Parcel SRM+LLM depth center of mass",
        "cmap": "inferno",
        "vmin": 0.0,
        "vmax": 100.0,
        "threshold": 0.001,
        "cbar_kind": "depth",
        "cbar_label": "Depth Center of Mass (%)",
        "symmetric": False,
        "description": "Positive-r-weighted depth center of mass across GPT-2 layers.",
    },
    {
        "map_id": "parcel_best_srm_group_encoding_r",
        "column": "best_srm_r",
        "title": "Parcel best SRM+LLM encoding r",
        "cmap": "viridis",
        "vmin": 0.0,
        "vmax": safe_vmax(preferred_df["best_srm_r"], percentile=99.0),
        "threshold": 0.00001,
        "cbar_kind": "r",
        "cbar_label": "Best SRM+LLM r",
        "symmetric": False,
        "description": "Best SRM+LLM group-level encoding performance for each parcel.",
    },
    {
        "map_id": "parcel_same_layer_srm_gain_at_best_srm_layer",
        "column": "same_layer_srm_gain_at_best_srm_layer",
        "title": "Parcel SRM gain at preferred layer",
        "cmap": "coolwarm",
        "vmin": -safe_absmax(preferred_df["same_layer_srm_gain_at_best_srm_layer"], percentile=99.0),
        "vmax": safe_absmax(preferred_df["same_layer_srm_gain_at_best_srm_layer"], percentile=99.0),
        "threshold": 0.00001,
        "cbar_kind": "decimal",
        "cbar_label": "SRM - Raw r",
        "symmetric": False,
        "description": "SRM+LLM minus Raw+LLM at each parcel's SRM-preferred layer.",
    },
    {
        "map_id": "parcel_preferred_layer_confidence",
        "column": "confidence_numeric",
        "title": "Parcel preferred-layer confidence",
        "cmap": "YlGnBu",
        "vmin": 1.0,
        "vmax": 3.0,
        "threshold": 0.5,
        "cbar_kind": "r",
        "cbar_label": "1 low | 2 moderate | 3 high",
        "symmetric": False,
        "description": "Preferred-layer confidence tier encoded as low/moderate/high.",
    },
]

# Add an interpretable-only preferred-depth map. Low-confidence parcels are left
# as background so later figures can focus on parcels with more stable estimates.
interpretable_df = preferred_df[preferred_df["srm_preferred_layer_is_interpretable"]].copy()
interpretable_spec = {
    "map_id": "parcel_best_srm_log_layer_depth_percent_interpretable_only",
    "column": "best_srm_log_depth_percent",
    "title": "Parcel preferred layer depth (interpretable only)",
    "cmap": "inferno",
    "vmin": 0.0,
    "vmax": 100.0,
    "threshold": 0.001,
    "cbar_kind": "depth",
    "cbar_label": "Best Encoding Layer (%)",
    "symmetric": False,
    "description": "Argmax preferred layer depth for moderate/high-confidence parcels only.",
    "source_df": "interpretable",
}
map_specs.append(interpretable_spec)

manifest_rows = []
for spec in map_specs:
    source_df = interpretable_df if spec.get("source_df") == "interpretable" else preferred_df
    value_map = source_df.set_index("parcel_id")[spec["column"]].to_dict()
    nii_path = NII_DIR / f"{spec['map_id']}.nii.gz"
    parcel_values_to_nifti(value_map, nii_path, background_value=0.0)

    glass_path = FIG_GLASS_DIR / f"{spec['map_id']}_glass.png"
    save_glass_map(
        img_path=nii_path,
        out_path=glass_path,
        title=spec["title"],
        cmap=spec["cmap"],
        vmin=spec["vmin"],
        vmax=spec["vmax"],
        threshold=spec["threshold"],
        cbar_kind=spec["cbar_kind"],
        cbar_label=spec["cbar_label"],
        symmetric_cbar=spec["symmetric"],
    )

    ortho_path = FIG_ORTHO_DIR / f"{spec['map_id']}_ortho.png"
    save_ortho_map(
        img_path=nii_path,
        out_path=ortho_path,
        title=spec["title"],
        cmap=spec["cmap"],
        vmin=spec["vmin"],
        vmax=spec["vmax"],
        threshold=spec["threshold"],
        cbar_kind=spec["cbar_kind"],
        cbar_label=spec["cbar_label"],
        symmetric_cbar=spec["symmetric"],
    )

    for axis, view_name, display_mode in [
        ("x", "sagittal", "x"),
        ("y", "coronal", "y"),
        ("z", "axial_horizontal", "z"),
    ]:
        short_id = short_map_id(spec["map_id"])
        direction_dir = FIG_XYZ_DIR / short_id
        direction_dir.mkdir(parents=True, exist_ok=True)
        xyz_path = direction_dir / f"{short_id}_{axis}.png"
        save_directional_map(
            img_path=nii_path,
            out_path=xyz_path,
            title=f"{spec['title']} ({view_name.replace('_', ' ')})",
            display_mode=display_mode,
            cmap=spec["cmap"],
            vmin=spec["vmin"],
            vmax=spec["vmax"],
            threshold=spec["threshold"],
            cbar_kind=spec["cbar_kind"],
            cbar_label=spec["cbar_label"],
            symmetric_cbar=spec["symmetric"],
        )
        manifest_rows.append({
            "map_id": spec["map_id"],
            "nifti_path": str(nii_path),
            "figure_type": "directional_slice",
            "axis": axis,
            "view_name": view_name,
            "figure_path": str(xyz_path),
            "column": spec["column"],
            "vmin": float(spec["vmin"]),
            "vmax": float(spec["vmax"]),
            "threshold": float(spec["threshold"]),
            "description": spec["description"],
        })

    manifest_rows.append({
        "map_id": spec["map_id"],
        "nifti_path": str(nii_path),
        "figure_type": "glass",
        "axis": "all",
        "view_name": "glass_lyrz",
        "figure_path": str(glass_path),
        "column": spec["column"],
        "vmin": float(spec["vmin"]),
        "vmax": float(spec["vmax"]),
        "threshold": float(spec["threshold"]),
        "description": spec["description"],
    })
    manifest_rows.append({
        "map_id": spec["map_id"],
        "nifti_path": str(nii_path),
        "figure_type": "ortho",
        "axis": "ortho",
        "view_name": "ortho_summary",
        "figure_path": str(ortho_path),
        "column": spec["column"],
        "vmin": float(spec["vmin"]),
        "vmax": float(spec["vmax"]),
        "threshold": float(spec["threshold"]),
        "description": spec["description"],
    })

# =========================
# 5. Save manifest and summary
# =========================

manifest_df = pd.DataFrame(manifest_rows)
manifest_csv = CSV_DIR / "step11C_parcel_preferred_layer_brain_map_manifest.csv"
manifest_json = JSON_DIR / "step11C_parcel_preferred_layer_brain_map_manifest.json"
manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
with open(manifest_json, "w", encoding="utf-8") as f:
    json.dump(manifest_rows, f, indent=2, ensure_ascii=False)

summary = {
    "step": "step11C_parcel_preferred_layer_brain_maps",
    "description": "Map parcel-level preferred GPT-2 layer depth, depth center of mass, encoding strength, SRM gain, and confidence back to brain space.",
    "input_preferred_table": str(PREFERRED_TABLE_PATH),
    "n_parcels": int(preferred_df["parcel_id"].nunique()),
    "n_interpretable_parcels": int(preferred_df["srm_preferred_layer_is_interpretable"].sum()),
    "n_maps": int(len(map_specs)),
    "maps": [spec["map_id"] for spec in map_specs],
    "output_dirs": {
        "nii_dir": str(NII_DIR),
        "glass_figures": str(FIG_GLASS_DIR),
        "ortho_figures": str(FIG_ORTHO_DIR),
        "directional_slice_figures": str(FIG_XYZ_DIR),
        "manifest_csv": str(manifest_csv),
    },
    "elapsed_minutes": float((time.time() - start_time) / 60.0),
}
with open(JSON_DIR / "step11C_parcel_preferred_layer_brain_map_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11C completed: parcel-level preferred-layer brain maps")
print(f"Saved NIfTI maps : {NII_DIR}")
print(f"Saved glass maps : {FIG_GLASS_DIR}")
print(f"Saved ortho maps : {FIG_ORTHO_DIR}")
print(f"Saved XYZ maps   : {FIG_XYZ_DIR}")
print(f"Saved manifest   : {manifest_csv}")
display(manifest_df.head(20))


In [ ]:
# Step 11D: Parcel-Level SRM Gain Metrics And Maps
# This step keeps the analysis at the Schaefer parcel level and asks where SRM improves layer-wise LLM encoding.
# For each parcel, it computes SRM-minus-Raw gain at the parcel's own preferred SRM layer, summarizes gain across layer-depth bins, and writes the main gain metrics back to atlas-space brain maps.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from nilearn.plotting import plot_glass_brain, plot_stat_map

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# =========================
# Step 11D setup
# =========================

start_time = time.time()

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
OUTPUT_ROOT = BASE_DIR / "parcel_layer_network_outputs"

STEP11A_DIR = OUTPUT_ROOT / "step11A_build_parcel_level_encoding_table"
STEP11B_DIR = OUTPUT_ROOT / "step11B_parcel_preferred_layer"
STEP11D_DIR = OUTPUT_ROOT / "step11D_parcel_level_srm_gain_maps"

CSV_DIR = STEP11D_DIR / "csv"
XLSX_DIR = STEP11D_DIR / "xlsx"
NII_DIR = STEP11D_DIR / "nii"
FIG_GLASS_DIR = STEP11D_DIR / "figures_glass"
FIG_ORTHO_DIR = STEP11D_DIR / "figures_ortho_summary"
FIG_XYZ_DIR = STEP11D_DIR / "figures_xyz_directional_slices"
JSON_DIR = STEP11D_DIR / "json"

for d in [STEP11D_DIR, CSV_DIR, XLSX_DIR, NII_DIR, FIG_GLASS_DIR, FIG_ORTHO_DIR, FIG_XYZ_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PARCEL_LAYER_TABLE_PATH = STEP11A_DIR / "csv" / "step11A_parcel_layer_encoding_table.csv"
PREFERRED_LAYER_TABLE_PATH = STEP11B_DIR / "csv" / "step11B_parcel_preferred_layer_table.csv"
ATLAS_PATH = Path(r"YOUR_SCHAEFER400_ATLAS_NIFTI_PATH")

for path in [PARCEL_LAYER_TABLE_PATH, PREFERRED_LAYER_TABLE_PATH, ATLAS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")

pd.options.display.float_format = lambda x: f"{x:.5f}"

print("Step 11D: Parcel-level SRM gain metrics and maps")
print(f"Parcel x layer table : {PARCEL_LAYER_TABLE_PATH}")
print(f"Preferred layer table: {PREFERRED_LAYER_TABLE_PATH}")
print(f"Atlas                : {ATLAS_PATH}")

# =========================
# Load and validate inputs
# =========================

parcel_layer_df = pd.read_csv(PARCEL_LAYER_TABLE_PATH)
preferred_df = pd.read_csv(PREFERRED_LAYER_TABLE_PATH)

required_layer_cols = [
    "parcel_id", "parcel_label", "roi_name", "network_family",
    "layer_index", "layer_label", "log_layer_depth_percent",
    "raw_mean_r", "srm_mean_r", "delta_mean_r",
]
missing_layer_cols = [c for c in required_layer_cols if c not in parcel_layer_df.columns]
if missing_layer_cols:
    raise ValueError(f"Step 11A table is missing required columns: {missing_layer_cols}")

required_pref_cols = [
    "parcel_id", "best_srm_layer_index", "best_srm_layer_label",
    "best_srm_log_depth_percent", "best_srm_depth_bin", "best_srm_r",
    "srm_depth_center_of_mass", "srm_preferred_layer_confidence",
    "srm_preferred_layer_is_interpretable",
]
missing_pref_cols = [c for c in required_pref_cols if c not in preferred_df.columns]
if missing_pref_cols:
    raise ValueError(f"Step 11B table is missing required columns: {missing_pref_cols}")

analysis_df = parcel_layer_df[parcel_layer_df["layer_index"] > 0].copy()
analysis_df["layer_index"] = analysis_df["layer_index"].astype(int)
preferred_df["parcel_id"] = preferred_df["parcel_id"].astype(int)
preferred_lookup = preferred_df.set_index("parcel_id").to_dict(orient="index")

for col in ["raw_mean_r", "srm_mean_r", "delta_mean_r", "log_layer_depth_percent"]:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

if analysis_df[["raw_mean_r", "srm_mean_r", "delta_mean_r", "log_layer_depth_percent"]].isna().any().any():
    bad = analysis_df[analysis_df[["raw_mean_r", "srm_mean_r", "delta_mean_r", "log_layer_depth_percent"]].isna().any(axis=1)].head(10)
    raise ValueError(f"Non-finite values detected in Step 11D inputs. First rows:\n{bad}")

def parse_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(x)
    if isinstance(x, str):
        return x.strip().lower() in {"true", "1", "yes", "y"}
    return False

def assign_depth_bin_3(log_depth_percent):
    if log_depth_percent < 50:
        return "early"
    if log_depth_percent < 90:
        return "middle"
    return "late"

analysis_df["depth_bin_3"] = analysis_df["log_layer_depth_percent"].apply(assign_depth_bin_3)
analysis_df["depth_bin_3"] = pd.Categorical(
    analysis_df["depth_bin_3"],
    categories=["early", "middle", "late"],
    ordered=True,
)
analysis_df["is_positive_gain"] = analysis_df["delta_mean_r"] > 0

print("Loaded Step 11A/11B inputs")
print(f"Transformer-layer rows: {len(analysis_df)}")
print(f"Parcels               : {analysis_df['parcel_id'].nunique()}")
print(f"Networks              : {analysis_df['roi_name'].nunique()}")
print("Layer-depth bin counts:")
print(
    analysis_df[["layer_index", "depth_bin_3"]]
    .drop_duplicates()
    .groupby("depth_bin_3", observed=True)
    .size()
    .to_string()
)

# =========================
# Compute parcel-level SRM gain metrics
# =========================

metadata_candidates = [
    "parcel_short_name", "hemisphere", "schaefer_network", "n_voxels_retained",
    "centroid_x", "centroid_y", "centroid_z",
]
metadata_cols = [c for c in metadata_candidates if c in analysis_df.columns]

parcel_rows = []

for parcel_id, parcel_df in analysis_df.groupby("parcel_id", sort=True):
    parcel_id = int(parcel_id)
    parcel_df = parcel_df.sort_values("layer_index").copy()
    first = parcel_df.iloc[0]
    pref = preferred_lookup.get(parcel_id, {})

    if not pref:
        raise ValueError(f"Missing Step 11B preferred-layer row for parcel_id={parcel_id}")

    best_srm_layer_index = int(pref["best_srm_layer_index"])
    best_layer_df = parcel_df[parcel_df["layer_index"] == best_srm_layer_index]
    if best_layer_df.empty:
        raise ValueError(f"Parcel {parcel_id} has no row for best_srm_layer_index={best_srm_layer_index}")

    best_layer_row = best_layer_df.iloc[0]
    same_layer_raw_r = float(best_layer_row["raw_mean_r"])
    same_layer_srm_r = float(best_layer_row["srm_mean_r"])
    same_layer_gain_from_table = float(best_layer_row["delta_mean_r"])
    same_layer_gain = same_layer_srm_r - same_layer_raw_r

    max_gain_row = parcel_df.loc[parcel_df["delta_mean_r"].idxmax()]

    row = {
        "parcel_id": parcel_id,
        "parcel_label": first["parcel_label"],
        "roi_name": first["roi_name"],
        "network_family": first["network_family"],
        "n_layers": int(parcel_df["layer_index"].nunique()),
        "best_srm_layer_index": best_srm_layer_index,
        "best_srm_layer_label": pref["best_srm_layer_label"],
        "best_srm_log_depth_percent": float(pref["best_srm_log_depth_percent"]),
        "best_srm_depth_bin": pref["best_srm_depth_bin"],
        "best_srm_r": float(pref["best_srm_r"]),
        "srm_depth_center_of_mass": float(pref["srm_depth_center_of_mass"]),
        "srm_preferred_layer_confidence": pref["srm_preferred_layer_confidence"],
        "srm_preferred_layer_is_interpretable": parse_bool(pref["srm_preferred_layer_is_interpretable"]),
        "same_layer_raw_r": same_layer_raw_r,
        "same_layer_srm_r": same_layer_srm_r,
        "same_layer_gain": same_layer_gain,
        "same_layer_gain_from_table": same_layer_gain_from_table,
        "same_layer_gain_rounding_diff": same_layer_gain - same_layer_gain_from_table,
        "max_layer_gain": float(max_gain_row["delta_mean_r"]),
        "max_gain_layer_index": int(max_gain_row["layer_index"]),
        "max_gain_layer_label": max_gain_row["layer_label"],
        "max_gain_log_depth_percent": float(max_gain_row["log_layer_depth_percent"]),
        "max_gain_depth_bin_3": str(max_gain_row["depth_bin_3"]),
        "overall_mean_raw_r": float(parcel_df["raw_mean_r"].mean()),
        "overall_mean_srm_r": float(parcel_df["srm_mean_r"].mean()),
        "overall_mean_gain": float(parcel_df["delta_mean_r"].mean()),
        "overall_positive_gain_fraction": float(parcel_df["is_positive_gain"].mean()),
    }

    for col in metadata_cols:
        row[col] = first[col]

    for bin_name in ["early", "middle", "late"]:
        bin_df = parcel_df[parcel_df["depth_bin_3"] == bin_name]
        row[f"{bin_name}_n_layers"] = int(bin_df["layer_index"].nunique())
        row[f"{bin_name}_mean_raw_r"] = float(bin_df["raw_mean_r"].mean())
        row[f"{bin_name}_mean_srm_r"] = float(bin_df["srm_mean_r"].mean())
        row[f"{bin_name}_mean_gain"] = float(bin_df["delta_mean_r"].mean())
        row[f"{bin_name}_positive_gain_fraction"] = float(bin_df["is_positive_gain"].mean())

    row["middle_minus_early_gain"] = row["middle_mean_gain"] - row["early_mean_gain"]
    row["late_minus_middle_gain"] = row["late_mean_gain"] - row["middle_mean_gain"]
    row["late_minus_early_gain"] = row["late_mean_gain"] - row["early_mean_gain"]

    contextual_df = parcel_df[parcel_df["depth_bin_3"].isin(["middle", "late"])]
    row["contextual_mean_gain"] = float(contextual_df["delta_mean_r"].mean())
    row["contextual_minus_early_gain"] = row["contextual_mean_gain"] - row["early_mean_gain"]
    row["contextual_positive_gain_fraction"] = float(contextual_df["is_positive_gain"].mean())

    y = np.array([row["early_mean_gain"], row["middle_mean_gain"], row["late_mean_gain"]], dtype=float)
    row["depth_gain_slope"] = float(np.polyfit(np.array([0.0, 1.0, 2.0]), y, 1)[0])
    row["late_gt_middle_gt_early"] = bool(row["late_mean_gain"] > row["middle_mean_gain"] > row["early_mean_gain"])

    parcel_rows.append(row)

parcel_gain_df = pd.DataFrame(parcel_rows)

max_rounding_diff = float(np.nanmax(np.abs(parcel_gain_df["same_layer_gain_rounding_diff"])))
if max_rounding_diff > 1e-4:
    raise ValueError(
        f"same_layer_gain differs from the stored delta_mean_r by more than rounding tolerance. "
        f"Max difference = {max_rounding_diff:.8f}"
    )
print(f"Max same-layer gain rounding difference vs stored delta_mean_r: {max_rounding_diff:.8f}")

front_cols = [
    "parcel_id", "parcel_label", "roi_name", "network_family",
    "best_srm_layer_index", "best_srm_layer_label", "best_srm_log_depth_percent", "best_srm_depth_bin",
    "best_srm_r", "same_layer_raw_r", "same_layer_srm_r", "same_layer_gain",
    "max_layer_gain", "early_mean_gain", "middle_mean_gain", "late_mean_gain",
    "late_minus_early_gain", "contextual_minus_early_gain", "depth_gain_slope",
    "srm_depth_center_of_mass", "srm_preferred_layer_confidence", "srm_preferred_layer_is_interpretable",
]
remaining_cols = [c for c in parcel_gain_df.columns if c not in front_cols]
parcel_gain_df = parcel_gain_df[front_cols + remaining_cols]

# =========================
# Summaries by network and family
# =========================

def fraction_true(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return float(np.mean(x.astype(bool)))

network_summary_df = (
    parcel_gain_df
    .groupby(["roi_name", "network_family"], observed=True)
    .agg(
        n_parcels=("parcel_id", "nunique"),
        n_interpretable_parcels=("srm_preferred_layer_is_interpretable", "sum"),
        mean_best_srm_r=("best_srm_r", "mean"),
        mean_same_layer_gain=("same_layer_gain", "mean"),
        mean_max_layer_gain=("max_layer_gain", "mean"),
        mean_early_gain=("early_mean_gain", "mean"),
        mean_middle_gain=("middle_mean_gain", "mean"),
        mean_late_gain=("late_mean_gain", "mean"),
        mean_late_minus_early_gain=("late_minus_early_gain", "mean"),
        mean_contextual_minus_early_gain=("contextual_minus_early_gain", "mean"),
        mean_depth_gain_slope=("depth_gain_slope", "mean"),
        fraction_late_gt_middle_gt_early=("late_gt_middle_gt_early", fraction_true),
        interpretable_parcel_fraction=("srm_preferred_layer_is_interpretable", fraction_true),
    )
    .reset_index()
    .sort_values(["mean_same_layer_gain", "mean_contextual_minus_early_gain"], ascending=False)
)

family_summary_df = (
    parcel_gain_df
    .groupby("network_family", observed=True)
    .agg(
        n_parcels=("parcel_id", "nunique"),
        n_networks=("roi_name", "nunique"),
        mean_best_srm_r=("best_srm_r", "mean"),
        mean_same_layer_gain=("same_layer_gain", "mean"),
        mean_max_layer_gain=("max_layer_gain", "mean"),
        mean_early_gain=("early_mean_gain", "mean"),
        mean_middle_gain=("middle_mean_gain", "mean"),
        mean_late_gain=("late_mean_gain", "mean"),
        mean_late_minus_early_gain=("late_minus_early_gain", "mean"),
        mean_contextual_minus_early_gain=("contextual_minus_early_gain", "mean"),
        mean_depth_gain_slope=("depth_gain_slope", "mean"),
        fraction_late_gt_middle_gt_early=("late_gt_middle_gt_early", fraction_true),
        interpretable_parcel_fraction=("srm_preferred_layer_is_interpretable", fraction_true),
    )
    .reset_index()
    .sort_values(["mean_same_layer_gain", "mean_contextual_minus_early_gain"], ascending=False)
)

top_same_layer_gain_df = parcel_gain_df.sort_values("same_layer_gain", ascending=False).head(50)
top_contextual_gain_df = parcel_gain_df.sort_values("contextual_minus_early_gain", ascending=False).head(50)
top_max_gain_df = parcel_gain_df.sort_values("max_layer_gain", ascending=False).head(50)

# =========================
# Save metric tables
# =========================

csv_outputs = {
    "step11D_parcel_srm_gain_metrics.csv": parcel_gain_df,
    "step11D_network_srm_gain_summary.csv": network_summary_df,
    "step11D_family_srm_gain_summary.csv": family_summary_df,
    "step11D_top_parcels_by_same_layer_gain.csv": top_same_layer_gain_df,
    "step11D_top_parcels_by_contextual_minus_early_gain.csv": top_contextual_gain_df,
    "step11D_top_parcels_by_max_layer_gain.csv": top_max_gain_df,
}

for name, df in csv_outputs.items():
    df.to_csv(CSV_DIR / name, index=False, encoding="utf-8-sig", float_format="%.5f")

xlsx_path = XLSX_DIR / "step11D_parcel_level_srm_gain_metrics_and_maps.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    parcel_gain_df.to_excel(writer, sheet_name="parcel_srm_gain_metrics", index=False)
    network_summary_df.to_excel(writer, sheet_name="network_summary", index=False)
    family_summary_df.to_excel(writer, sheet_name="family_summary", index=False)
    top_same_layer_gain_df.to_excel(writer, sheet_name="top_same_layer_gain", index=False)
    top_contextual_gain_df.to_excel(writer, sheet_name="top_contextual_gain", index=False)
    top_max_gain_df.to_excel(writer, sheet_name="top_max_gain", index=False)

print("\nSaved Step 11D metric tables")
print(f"CSV directory : {CSV_DIR}")
print(f"Excel workbook: {xlsx_path}")

# =========================
# Map parcel values back to atlas-space NIfTI
# =========================

atlas_nii = nib.load(str(ATLAS_PATH))
atlas_data = np.asanyarray(atlas_nii.get_fdata(), dtype=float)
atlas_labels = np.rint(atlas_data).astype(np.int32)
valid_atlas_labels = set(int(v) for v in np.unique(atlas_labels) if v > 0)
missing_atlas_labels = sorted(set(parcel_gain_df["parcel_id"].astype(int)) - valid_atlas_labels)
if missing_atlas_labels:
    raise ValueError(f"Parcel IDs not found in Schaefer atlas: {missing_atlas_labels[:10]}")

def parcel_values_to_nifti(values_by_parcel, out_path):
    out = np.zeros(atlas_labels.shape, dtype=np.float32)
    for parcel_id, value in values_by_parcel.items():
        if pd.isna(value):
            continue
        out[atlas_labels == int(parcel_id)] = np.float32(value)
    nii = nib.Nifti1Image(out, atlas_nii.affine, atlas_nii.header)
    nib.save(nii, str(out_path))
    return out_path

def finite_nonzero_values(img_path, eps=1e-8):
    data = np.asanyarray(nib.load(str(img_path)).get_fdata(), dtype=float)
    return data[np.isfinite(data) & (np.abs(data) > eps)]

def robust_absmax(values, percentile=99):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = np.abs(values[values != 0])
    if values.size == 0:
        return 0.00001
    vmax = float(np.nanpercentile(values, percentile))
    if (not np.isfinite(vmax)) or vmax <= 0:
        return 0.00001
    return vmax

def robust_posmax(values, percentile=99):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[values > 0]
    if values.size == 0:
        return 0.00001
    vmax = float(np.nanpercentile(values, percentile))
    if (not np.isfinite(vmax)) or vmax <= 0:
        return 0.00001
    return vmax

def format_colorbar(disp, fmt):
    try:
        disp._cbar.ax.yaxis.set_major_formatter(FormatStrFormatter(fmt))
        disp._cbar.ax.xaxis.set_major_formatter(FormatStrFormatter(fmt))
        disp._cbar.ax.yaxis.offsetText.set_visible(False)
        disp._cbar.ax.xaxis.offsetText.set_visible(False)
        try:
            disp._cbar.formatter.set_scientific(False)
            disp._cbar.formatter.set_useOffset(False)
        except Exception:
            pass
        disp._cbar.update_ticks()
    except Exception:
        pass

def safe_cut_coords(img_path, eps=1e-8):
    img = nib.load(str(img_path))
    data = np.asarray(img.get_fdata(), dtype=float)
    valid = np.isfinite(data) & (np.abs(data) > eps)
    if np.any(valid):
        abs_data = np.zeros(data.shape, dtype=float)
        abs_data[valid] = np.abs(data[valid])
        valid_abs = abs_data[valid]
        cutoff = np.nanpercentile(valid_abs, 95)
        strong = valid & (abs_data >= cutoff)
        if np.any(strong):
            ijk_mean = np.mean(np.argwhere(strong), axis=0)
            xyz = nib.affines.apply_affine(img.affine, ijk_mean)
            return tuple(float(v) for v in xyz)
    return (0.0, 0.0, 0.0)

map_specs = [
    {
        "map_id": "parcel_same_layer_srm_gain",
        "column": "same_layer_gain",
        "title": "Parcel same-layer SRM gain",
        "description": "SRM - Raw encoding r at each parcel's preferred SRM layer.",
        "cmap": "coolwarm",
        "scale": "symmetric",
        "fmt": "%.5f",
    },
    {
        "map_id": "parcel_max_layer_srm_gain",
        "column": "max_layer_gain",
        "title": "Parcel maximum SRM gain",
        "description": "Maximum SRM - Raw encoding r across GPT-2 transformer layers for each parcel.",
        "cmap": "coolwarm",
        "scale": "symmetric",
        "fmt": "%.5f",
    },
    {
        "map_id": "parcel_contextual_minus_early_srm_gain",
        "column": "contextual_minus_early_gain",
        "title": "Parcel contextual minus early SRM gain",
        "description": "Mean middle/late SRM gain minus early SRM gain for each parcel.",
        "cmap": "coolwarm",
        "scale": "symmetric",
        "fmt": "%.5f",
    },
    {
        "map_id": "parcel_late_minus_early_srm_gain",
        "column": "late_minus_early_gain",
        "title": "Parcel late minus early SRM gain",
        "description": "Late-layer SRM gain minus early-layer SRM gain for each parcel.",
        "cmap": "coolwarm",
        "scale": "symmetric",
        "fmt": "%.5f",
    },
    {
        "map_id": "parcel_best_srm_encoding_r",
        "column": "best_srm_r",
        "title": "Parcel best SRM+LLM encoding r",
        "description": "Best SRM-reconstructed encoding r across GPT-2 transformer layers for each parcel.",
        "cmap": "viridis",
        "scale": "positive",
        "fmt": "%.5f",
    },
]

nii_rows = []
for spec in map_specs:
    values_by_parcel = parcel_gain_df.set_index("parcel_id")[spec["column"]].to_dict()
    nii_path = NII_DIR / f"{spec['map_id']}.nii.gz"
    parcel_values_to_nifti(values_by_parcel, nii_path)
    vals = finite_nonzero_values(nii_path)
    if spec["scale"] == "symmetric":
        vmax = robust_absmax(vals, percentile=99)
        vmin = -vmax
    else:
        vmin = 0.0
        vmax = robust_posmax(vals, percentile=99)
    threshold = 1e-8
    nii_rows.append({
        "map_id": spec["map_id"],
        "metric_column": spec["column"],
        "nifti_path": str(nii_path),
        "vmin": float(vmin),
        "vmax": float(vmax),
        "threshold": float(threshold),
        "cmap": spec["cmap"],
        "scale": spec["scale"],
        "colorbar_format": spec["fmt"],
        "description": spec["description"],
    })

nii_manifest_df = pd.DataFrame(nii_rows)
nii_manifest_df.to_csv(CSV_DIR / "step11D_parcel_srm_gain_nifti_manifest.csv", index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nSaved parcel-level SRM gain NIfTI maps")
print(nii_manifest_df[["map_id", "nifti_path", "vmin", "vmax"]].to_string(index=False))

# =========================
# Plot gain maps
# =========================

plot_rows = []

def save_glass_map(spec, nii_path, vmin, vmax, threshold):
    out_path = FIG_GLASS_DIR / f"{spec['map_id']}_glass.png"
    disp = plot_glass_brain(
        str(nii_path),
        display_mode="lyrz",
        colorbar=True,
        cmap=spec["cmap"],
        plot_abs=False,
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        black_bg=False,
        title=spec["title"],
    )
    format_colorbar(disp, spec["fmt"])
    disp.savefig(out_path, dpi=260)
    plt.close()
    return out_path

def save_ortho_map(spec, nii_path, vmin, vmax, threshold):
    out_path = FIG_ORTHO_DIR / f"{spec['map_id']}_ortho_summary.png"
    cut_coords = safe_cut_coords(nii_path)
    disp = plot_stat_map(
        str(nii_path),
        display_mode="ortho",
        cut_coords=cut_coords,
        colorbar=True,
        cmap=spec["cmap"],
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        black_bg=False,
        dim=-0.4,
        draw_cross=True,
        symmetric_cbar=False,
        cbar_tick_format=spec["fmt"],
        title=spec["title"],
    )
    format_colorbar(disp, spec["fmt"])
    disp.savefig(out_path, dpi=280, bbox_inches="tight", pad_inches=0.05)
    plt.close()
    return out_path, cut_coords

def save_directional_map(spec, nii_path, vmin, vmax, threshold, axis, display_mode, view_name):
    out_dir = FIG_XYZ_DIR / spec["map_id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{spec['map_id']}_{axis}_{view_name}.png"
    disp = plot_stat_map(
        str(nii_path),
        display_mode=display_mode,
        cut_coords=7,
        colorbar=True,
        cmap=spec["cmap"],
        threshold=threshold,
        vmin=vmin,
        vmax=vmax,
        black_bg=False,
        symmetric_cbar=False,
        cbar_tick_format=spec["fmt"],
        title=f"{spec['title']} ({view_name.replace('_', ' ')})",
    )
    format_colorbar(disp, spec["fmt"])
    disp.savefig(out_path, dpi=260, bbox_inches="tight", pad_inches=0.05)
    plt.close()
    return out_path

spec_lookup = {s["map_id"]: s for s in map_specs}
directions = [
    ("x", "x", "sagittal"),
    ("y", "y", "coronal"),
    ("z", "z", "axial_horizontal"),
]

for _, row in nii_manifest_df.iterrows():
    spec = spec_lookup[row["map_id"]]
    nii_path = Path(row["nifti_path"])
    vmin = float(row["vmin"])
    vmax = float(row["vmax"])
    threshold = float(row["threshold"])

    glass_path = save_glass_map(spec, nii_path, vmin, vmax, threshold)
    plot_rows.append({
        "map_id": spec["map_id"],
        "figure_type": "glass",
        "figure_path": str(glass_path),
        "nifti_path": str(nii_path),
        "vmin": vmin,
        "vmax": vmax,
        "threshold": threshold,
        "description": spec["description"],
    })

    ortho_path, cut_coords = save_ortho_map(spec, nii_path, vmin, vmax, threshold)
    plot_rows.append({
        "map_id": spec["map_id"],
        "figure_type": "ortho",
        "figure_path": str(ortho_path),
        "nifti_path": str(nii_path),
        "cut_x": float(cut_coords[0]),
        "cut_y": float(cut_coords[1]),
        "cut_z": float(cut_coords[2]),
        "vmin": vmin,
        "vmax": vmax,
        "threshold": threshold,
        "description": spec["description"],
    })

    for axis, display_mode, view_name in directions:
        directional_path = save_directional_map(spec, nii_path, vmin, vmax, threshold, axis, display_mode, view_name)
        plot_rows.append({
            "map_id": spec["map_id"],
            "figure_type": f"directional_{axis}",
            "figure_path": str(directional_path),
            "nifti_path": str(nii_path),
            "axis": axis,
            "view_name": view_name,
            "vmin": vmin,
            "vmax": vmax,
            "threshold": threshold,
            "description": spec["description"],
        })

plot_manifest_df = pd.DataFrame(plot_rows)
plot_manifest_df.to_csv(CSV_DIR / "step11D_parcel_srm_gain_figure_manifest.csv", index=False, encoding="utf-8-sig", float_format="%.5f")

print("\nSaved parcel-level SRM gain figures")
print(f"Glass figures      : {FIG_GLASS_DIR}")
print(f"Ortho figures      : {FIG_ORTHO_DIR}")
print(f"Directional figures: {FIG_XYZ_DIR}")

# =========================
# Summary figures for tables
# =========================

fig, ax = plt.subplots(figsize=(10, 8))
plot_df = network_summary_df.sort_values("mean_same_layer_gain", ascending=True)
ax.barh(plot_df["roi_name"], plot_df["mean_same_layer_gain"], color="#D95F02", alpha=0.85)
ax.axvline(0, color="#444444", linestyle="--", linewidth=1)
ax.set_title("Network Mean Parcel Same-Layer SRM Gain")
ax.set_xlabel("Mean SRM - Raw encoding r")
ax.set_ylabel("Network")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
plt.tight_layout()
fig.savefig(FIG_GLASS_DIR / "step11D_network_same_layer_gain_barplot.png", dpi=300, bbox_inches="tight")
plt.close(fig)

network_order = network_summary_df.sort_values("mean_same_layer_gain", ascending=False)["roi_name"].tolist()
network_long_df = network_summary_df.melt(
    id_vars=["roi_name", "network_family", "n_parcels"],
    value_vars=["mean_early_gain", "mean_middle_gain", "mean_late_gain"],
    var_name="depth_bin",
    value_name="mean_gain",
)
network_long_df["depth_bin"] = network_long_df["depth_bin"].map({
    "mean_early_gain": "early",
    "mean_middle_gain": "middle",
    "mean_late_gain": "late",
})
network_long_df["depth_bin"] = pd.Categorical(network_long_df["depth_bin"], categories=["early", "middle", "late"], ordered=True)
network_long_df["roi_name"] = pd.Categorical(network_long_df["roi_name"], categories=network_order, ordered=True)

fig, ax = plt.subplots(figsize=(15, 7))
if HAS_SEABORN:
    sns.barplot(
        data=network_long_df,
        x="roi_name",
        y="mean_gain",
        hue="depth_bin",
        hue_order=["early", "middle", "late"],
        palette={"early": "#8DA0CB", "middle": "#66C2A5", "late": "#FC8D62"},
        ax=ax,
    )
else:
    pivot = network_long_df.pivot(index="roi_name", columns="depth_bin", values="mean_gain").loc[network_order]
    pivot.plot(kind="bar", ax=ax, color=["#8DA0CB", "#66C2A5", "#FC8D62"])
ax.axhline(0, color="#444444", linestyle="--", linewidth=1)
ax.set_title("Network Parcel SRM Gain by Layer-Depth Bin")
ax.set_xlabel("Network")
ax.set_ylabel("Mean SRM - Raw encoding r")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
ax.tick_params(axis="x", labelrotation=45)
ax.legend(title="Layer-depth bin", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(FIG_GLASS_DIR / "step11D_network_depth_bin_gain_barplot.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# =========================
# JSON summary and validation
# =========================

summary = {
    "step": "step11D_parcel_level_srm_gain_metrics_and_maps",
    "analysis_goal": "Compute parcel-level SRM-minus-Raw gain metrics and write the main gain metrics back to atlas-space brain maps.",
    "n_parcels": int(parcel_gain_df["parcel_id"].nunique()),
    "n_networks": int(parcel_gain_df["roi_name"].nunique()),
    "n_transformer_layers": int(analysis_df["layer_index"].nunique()),
    "mean_same_layer_gain": float(parcel_gain_df["same_layer_gain"].mean()),
    "mean_max_layer_gain": float(parcel_gain_df["max_layer_gain"].mean()),
    "mean_early_gain": float(parcel_gain_df["early_mean_gain"].mean()),
    "mean_middle_gain": float(parcel_gain_df["middle_mean_gain"].mean()),
    "mean_late_gain": float(parcel_gain_df["late_mean_gain"].mean()),
    "mean_late_minus_early_gain": float(parcel_gain_df["late_minus_early_gain"].mean()),
    "mean_contextual_minus_early_gain": float(parcel_gain_df["contextual_minus_early_gain"].mean()),
    "top_networks_by_same_layer_gain": network_summary_df.head(10).to_dict(orient="records"),
    "outputs": {
        "csv_dir": str(CSV_DIR),
        "xlsx_path": str(xlsx_path),
        "nii_dir": str(NII_DIR),
        "figure_glass_dir": str(FIG_GLASS_DIR),
        "figure_ortho_dir": str(FIG_ORTHO_DIR),
        "figure_xyz_dir": str(FIG_XYZ_DIR),
    },
    "elapsed_minutes": float((time.time() - start_time) / 60),
}

summary_path = JSON_DIR / "step11D_parcel_level_srm_gain_maps_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11D validation")
print(f"parcel_gain_df shape : {parcel_gain_df.shape}")
print(f"network_summary shape: {network_summary_df.shape}")
print(f"family_summary shape : {family_summary_df.shape}")
print(f"NIfTI maps           : {len(nii_manifest_df)}")
print(f"Figure rows          : {len(plot_manifest_df)}")
print(f"Mean same-layer gain : {parcel_gain_df['same_layer_gain'].mean():.5f}")
print(f"Mean max-layer gain  : {parcel_gain_df['max_layer_gain'].mean():.5f}")
print(f"Mean early gain      : {parcel_gain_df['early_mean_gain'].mean():.5f}")
print(f"Mean middle gain     : {parcel_gain_df['middle_mean_gain'].mean():.5f}")
print(f"Mean late gain       : {parcel_gain_df['late_mean_gain'].mean():.5f}")
print(f"Summary JSON         : {summary_path}")

show_cols = [
    "roi_name", "network_family", "n_parcels", "mean_best_srm_r",
    "mean_same_layer_gain", "mean_max_layer_gain", "mean_contextual_minus_early_gain",
]
print("\nTop networks by same-layer SRM gain:")
print(network_summary_df[show_cols].head(12).to_string(index=False))

display(network_summary_df[show_cols].head(12))
display(parcel_gain_df[front_cols].head(20))

print(f"\nTotal elapsed: {(time.time() - start_time) / 60:.2f} min")
print("Step 11D completed successfully.")



In [ ]:
# Step 11E: Within-Network Heterogeneity Analysis
# This step asks whether parcel-level GPT-2 layer preference is homogeneous or heterogeneous within each functional network.
# It uses the same four-bin log-depth scheme as the layer-depth brain maps: early, early-middle, middle-late, and late.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# =========================
# Step 11E setup
# =========================

start_time = time.time()

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
OUTPUT_ROOT = BASE_DIR / "parcel_layer_network_outputs"

STEP11D_DIR = OUTPUT_ROOT / "step11D_parcel_level_srm_gain_maps"
STEP11E_DIR = OUTPUT_ROOT / "step11E_within_network_heterogeneity"

CSV_DIR = STEP11E_DIR / "csv"
XLSX_DIR = STEP11E_DIR / "xlsx"
FIG_DIR = STEP11E_DIR / "figures"
JSON_DIR = STEP11E_DIR / "json"

for d in [STEP11E_DIR, CSV_DIR, XLSX_DIR, FIG_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PARCEL_GAIN_PATH = STEP11D_DIR / "csv" / "step11D_parcel_srm_gain_metrics.csv"

if not PARCEL_GAIN_PATH.exists():
    raise FileNotFoundError(f"Missing required Step 11D input: {PARCEL_GAIN_PATH}")

pd.options.display.float_format = lambda x: f"{x:.5f}"

print("Step 11E: Within-network heterogeneity analysis")
print(f"Parcel gain table: {PARCEL_GAIN_PATH}")

# =========================
# Load and validate inputs
# =========================

parcel_df = pd.read_csv(PARCEL_GAIN_PATH)

required_cols = [
    "parcel_id", "parcel_label", "roi_name", "network_family",
    "best_srm_log_depth_percent", "best_srm_depth_bin", "best_srm_r",
    "srm_depth_center_of_mass", "same_layer_gain", "max_layer_gain",
    "contextual_minus_early_gain", "late_minus_early_gain",
    "srm_preferred_layer_confidence", "srm_preferred_layer_is_interpretable",
]
missing_cols = [c for c in required_cols if c not in parcel_df.columns]
if missing_cols:
    raise ValueError(f"Step 11D parcel table is missing required columns: {missing_cols}")

critical_numeric_cols = [
    "best_srm_log_depth_percent", "best_srm_r",
    "same_layer_gain", "max_layer_gain", "contextual_minus_early_gain",
    "late_minus_early_gain",
]
optional_numeric_cols = ["srm_depth_center_of_mass"]
numeric_cols = critical_numeric_cols + optional_numeric_cols

for col in numeric_cols:
    parcel_df[col] = pd.to_numeric(parcel_df[col], errors="coerce")

if parcel_df[critical_numeric_cols].isna().any().any():
    bad = parcel_df[parcel_df[critical_numeric_cols].isna().any(axis=1)].head(10)
    raise ValueError(f"Non-finite critical values detected in Step 11E inputs. First rows:\n{bad}")

n_missing_com = int(parcel_df["srm_depth_center_of_mass"].isna().sum())
print(f"Optional missing srm_depth_center_of_mass values: {n_missing_com}")

def parse_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(x)
    if isinstance(x, str):
        return x.strip().lower() in {"true", "1", "yes", "y"}
    return False

parcel_df["srm_preferred_layer_is_interpretable"] = parcel_df["srm_preferred_layer_is_interpretable"].apply(parse_bool)

DEPTH_BIN_ORDER = ["early", "early_middle", "middle_late", "late"]
DEPTH_BIN_LABELS = {
    "early": "early",
    "early_middle": "early-middle",
    "middle_late": "middle-late",
    "late": "late",
}
DEPTH_BIN_COLORS = {
    "early": "#8DA0CB",
    "early_middle": "#66C2A5",
    "middle_late": "#FC8D62",
    "late": "#E78AC3",
}

# Same log-depth bins used in the layer-depth maps:
# early: <50%; early-middle: 50-72%; middle-late: 72-90%; late: >=90%.
def assign_log_depth_bin_4(depth_percent):
    if depth_percent < 50:
        return "early"
    if depth_percent < 72:
        return "early_middle"
    if depth_percent < 90:
        return "middle_late"
    return "late"

parcel_df["preferred_log_depth_bin"] = parcel_df["best_srm_log_depth_percent"].apply(assign_log_depth_bin_4)
parcel_df["preferred_log_depth_bin"] = pd.Categorical(
    parcel_df["preferred_log_depth_bin"],
    categories=DEPTH_BIN_ORDER,
    ordered=True,
)
parcel_df["preferred_depth_bin_label"] = parcel_df["preferred_log_depth_bin"].map(DEPTH_BIN_LABELS)

print("Loaded parcel-level SRM gain table")
print(f"Parcels: {parcel_df['parcel_id'].nunique()}")
print(f"Networks: {parcel_df['roi_name'].nunique()}")
print(f"Interpretable parcels: {int(parcel_df['srm_preferred_layer_is_interpretable'].sum())}")
print("Preferred log-depth bins:")
print(parcel_df["preferred_log_depth_bin"].value_counts().reindex(DEPTH_BIN_ORDER, fill_value=0).to_string())

# =========================
# Helper functions
# =========================

def safe_sd(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

def safe_iqr(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.percentile(x, 75) - np.percentile(x, 25))

def normalized_entropy(proportions):
    p = np.asarray(proportions, dtype=float)
    p = p[np.isfinite(p) & (p > 0)]
    if p.size <= 1:
        return 0.0
    ent = -np.sum(p * np.log(p))
    return float(ent / np.log(len(DEPTH_BIN_ORDER)))

def summarize_network(sub_df, suffix=""):
    n = int(sub_df["parcel_id"].nunique())
    depth = sub_df["best_srm_log_depth_percent"].astype(float).values
    com = sub_df["srm_depth_center_of_mass"].astype(float).values
    counts = sub_df["preferred_log_depth_bin"].value_counts(normalize=True).reindex(DEPTH_BIN_ORDER, fill_value=0.0)
    proportions = [float(counts.get(bin_name, 0.0)) for bin_name in DEPTH_BIN_ORDER]
    dominant_idx = int(np.argmax(proportions)) if proportions else 0
    dominant_bin = DEPTH_BIN_ORDER[dominant_idx]
    dominant_prop = float(np.max(proportions)) if proportions else np.nan
    depth_sd = safe_sd(depth)
    depth_iqr = safe_iqr(depth)
    depth_range = float(np.nanmax(depth) - np.nanmin(depth)) if len(depth) else np.nan
    entropy = normalized_entropy(proportions)
    heterogeneity_index = float((depth_sd / 50.0 + entropy) / 2.0)

    row = {
        f"n_parcels{suffix}": n,
        f"mean_preferred_depth{suffix}": float(np.nanmean(depth)),
        f"sd_preferred_depth{suffix}": depth_sd,
        f"iqr_preferred_depth{suffix}": depth_iqr,
        f"range_preferred_depth{suffix}": depth_range,
        f"mean_depth_center_of_mass{suffix}": float(np.nanmean(com)) if np.isfinite(com).any() else np.nan,
        f"sd_depth_center_of_mass{suffix}": safe_sd(com),
        f"dominant_depth_bin{suffix}": dominant_bin,
        f"dominant_depth_bin_label{suffix}": DEPTH_BIN_LABELS[dominant_bin],
        f"dominant_depth_bin_fraction{suffix}": dominant_prop,
        f"depth_bin_entropy{suffix}": entropy,
        f"heterogeneity_index{suffix}": heterogeneity_index,
        f"mean_best_srm_r{suffix}": float(sub_df["best_srm_r"].mean()),
        f"sd_best_srm_r{suffix}": safe_sd(sub_df["best_srm_r"].values),
        f"mean_same_layer_gain{suffix}": float(sub_df["same_layer_gain"].mean()),
        f"sd_same_layer_gain{suffix}": safe_sd(sub_df["same_layer_gain"].values),
        f"mean_max_layer_gain{suffix}": float(sub_df["max_layer_gain"].mean()),
        f"mean_contextual_minus_early_gain{suffix}": float(sub_df["contextual_minus_early_gain"].mean()),
        f"mean_late_minus_early_gain{suffix}": float(sub_df["late_minus_early_gain"].mean()),
    }
    for bin_name in DEPTH_BIN_ORDER:
        row[f"{bin_name}_parcel_proportion{suffix}"] = float(counts.get(bin_name, 0.0))
    return row

# =========================
# Compute heterogeneity summaries
# =========================

network_rows = []
for (roi_name, network_family), sub_df in parcel_df.groupby(["roi_name", "network_family"], observed=True):
    all_summary = summarize_network(sub_df, suffix="")
    interp_df = sub_df[sub_df["srm_preferred_layer_is_interpretable"]].copy()
    interp_summary = summarize_network(interp_df, suffix="_interpretable") if len(interp_df) else {}
    row = {
        "roi_name": roi_name,
        "network_family": network_family,
        **all_summary,
        **interp_summary,
    }
    network_rows.append(row)

network_heterogeneity_df = pd.DataFrame(network_rows)
network_heterogeneity_df = network_heterogeneity_df.sort_values(
    ["heterogeneity_index", "sd_preferred_depth"],
    ascending=False,
).reset_index(drop=True)

family_rows = []
for network_family, sub_df in parcel_df.groupby("network_family", observed=True):
    row = {
        "network_family": network_family,
        "n_networks": int(sub_df["roi_name"].nunique()),
        **summarize_network(sub_df, suffix=""),
    }
    family_rows.append(row)

family_heterogeneity_df = pd.DataFrame(family_rows)
family_heterogeneity_df = family_heterogeneity_df.sort_values(
    ["heterogeneity_index", "sd_preferred_depth"],
    ascending=False,
).reset_index(drop=True)

parcel_with_network_df = parcel_df.merge(
    network_heterogeneity_df[[
        "roi_name", "mean_preferred_depth", "sd_preferred_depth",
        "dominant_depth_bin", "dominant_depth_bin_label", "dominant_depth_bin_fraction",
        "depth_bin_entropy", "heterogeneity_index",
    ]],
    on="roi_name",
    how="left",
    suffixes=("", "_network"),
)
parcel_with_network_df["depth_deviation_from_network_mean"] = (
    parcel_with_network_df["best_srm_log_depth_percent"] -
    parcel_with_network_df["mean_preferred_depth"]
)
parcel_with_network_df["abs_depth_deviation_from_network_mean"] = parcel_with_network_df["depth_deviation_from_network_mean"].abs()

top_heterogeneous_networks_df = network_heterogeneity_df.head(10).copy()
top_consistent_networks_df = network_heterogeneity_df.sort_values(
    ["heterogeneity_index", "sd_preferred_depth"],
    ascending=True,
).head(10).copy()
top_outlier_parcels_df = parcel_with_network_df.sort_values(
    "abs_depth_deviation_from_network_mean",
    ascending=False,
).head(50).copy()

# =========================
# Save tables
# =========================

csv_outputs = {
    "step11E_network_within_network_heterogeneity_summary.csv": network_heterogeneity_df,
    "step11E_family_within_network_heterogeneity_summary.csv": family_heterogeneity_df,
    "step11E_parcel_depth_deviation_from_network_mean.csv": parcel_with_network_df,
    "step11E_top_heterogeneous_networks.csv": top_heterogeneous_networks_df,
    "step11E_top_consistent_networks.csv": top_consistent_networks_df,
    "step11E_top_depth_outlier_parcels.csv": top_outlier_parcels_df,
}

for name, df in csv_outputs.items():
    df.to_csv(CSV_DIR / name, index=False, encoding="utf-8-sig", float_format="%.5f")

xlsx_path = XLSX_DIR / "step11E_within_network_heterogeneity_tables.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    network_heterogeneity_df.to_excel(writer, sheet_name="network_heterogeneity", index=False)
    family_heterogeneity_df.to_excel(writer, sheet_name="family_heterogeneity", index=False)
    parcel_with_network_df.to_excel(writer, sheet_name="parcel_depth_deviation", index=False)
    top_heterogeneous_networks_df.to_excel(writer, sheet_name="top_heterogeneous", index=False)
    top_consistent_networks_df.to_excel(writer, sheet_name="top_consistent", index=False)
    top_outlier_parcels_df.to_excel(writer, sheet_name="top_outlier_parcels", index=False)

print("\nSaved Step 11E tables")
print(f"CSV directory : {CSV_DIR}")
print(f"Excel workbook: {xlsx_path}")

# =========================
# Figures
# =========================

network_order = network_heterogeneity_df.sort_values("heterogeneity_index", ascending=False)["roi_name"].tolist()

fig, ax = plt.subplots(figsize=(15, 7))
plot_df = parcel_df.copy()
plot_df["roi_name"] = pd.Categorical(plot_df["roi_name"], categories=network_order, ordered=True)
if HAS_SEABORN:
    sns.stripplot(
        data=plot_df,
        x="roi_name",
        y="best_srm_log_depth_percent",
        hue="preferred_log_depth_bin",
        hue_order=DEPTH_BIN_ORDER,
        palette=DEPTH_BIN_COLORS,
        dodge=False,
        jitter=0.22,
        alpha=0.75,
        size=5,
        ax=ax,
    )
    sns.pointplot(
        data=plot_df,
        x="roi_name",
        y="best_srm_log_depth_percent",
        color="#222222",
        errorbar="sd",
        join=False,
        markers="_",
        scale=1.2,
        ax=ax,
    )
else:
    for i, roi in enumerate(network_order):
        vals = plot_df.loc[plot_df["roi_name"] == roi, "best_srm_log_depth_percent"].values
        ax.scatter(np.full_like(vals, i, dtype=float), vals, alpha=0.7)
        ax.errorbar(i, np.mean(vals), yerr=safe_sd(vals), color="#222222", marker="_", capsize=4)
    ax.set_xticks(range(len(network_order)))
    ax.set_xticklabels(network_order)
ax.axhline(50, color="#777777", linestyle="--", linewidth=1)
ax.axhline(72, color="#777777", linestyle="--", linewidth=1)
ax.axhline(90, color="#777777", linestyle="--", linewidth=1)
ax.set_title("Within-Network Parcel Preferred Layer Depth")
ax.set_xlabel("Network")
ax.set_ylabel("Best SRM layer depth (%)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))
ax.tick_params(axis="x", labelrotation=45)
if HAS_SEABORN:
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    label_map = {k: DEPTH_BIN_LABELS.get(k, k) for k in unique.keys()}
    ax.legend(unique.values(), [label_map[k] for k in unique.keys()], title="Preferred log-depth bin", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(FIG_DIR / "step11E_network_preferred_depth_stripplot.png", dpi=300, bbox_inches="tight")
plt.close(fig)

prop_cols = [f"{bin_name}_parcel_proportion" for bin_name in DEPTH_BIN_ORDER]
prop_df = network_heterogeneity_df[["roi_name"] + prop_cols].copy()
prop_df["roi_name"] = pd.Categorical(prop_df["roi_name"], categories=network_order, ordered=True)
prop_df = prop_df.sort_values("roi_name")
fig, ax = plt.subplots(figsize=(15, 6.5))
bottom = np.zeros(len(prop_df))
for bin_name, col in zip(DEPTH_BIN_ORDER, prop_cols):
    vals = prop_df[col].values
    ax.bar(prop_df["roi_name"].astype(str), vals, bottom=bottom, label=DEPTH_BIN_LABELS[bin_name], color=DEPTH_BIN_COLORS[bin_name], alpha=0.9)
    bottom += vals
ax.set_title("Within-Network Preferred Depth-Bin Proportions")
ax.set_xlabel("Network")
ax.set_ylabel("Parcel proportion")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.tick_params(axis="x", labelrotation=45)
ax.legend(title="Preferred log-depth bin", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(FIG_DIR / "step11E_network_depth_bin_proportions.png", dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 8))
bar_df = network_heterogeneity_df.sort_values("heterogeneity_index", ascending=True)
ax.barh(bar_df["roi_name"], bar_df["heterogeneity_index"], color="#7570B3", alpha=0.85)
ax.set_title("Within-Network Layer-Depth Heterogeneity")
ax.set_xlabel("Heterogeneity index")
ax.set_ylabel("Network")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
plt.tight_layout()
fig.savefig(FIG_DIR / "step11E_network_heterogeneity_index_barplot.png", dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 6.5))
if HAS_SEABORN:
    sns.scatterplot(
        data=network_heterogeneity_df,
        x="heterogeneity_index",
        y="mean_best_srm_r",
        hue="network_family",
        size="mean_same_layer_gain",
        sizes=(70, 240),
        alpha=0.9,
        edgecolor="white",
        linewidth=0.8,
        ax=ax,
    )
else:
    ax.scatter(network_heterogeneity_df["heterogeneity_index"], network_heterogeneity_df["mean_best_srm_r"], s=90, alpha=0.8)
for _, row in network_heterogeneity_df.iterrows():
    ax.text(row["heterogeneity_index"] + 0.005, row["mean_best_srm_r"], row["roi_name"], fontsize=8)
ax.set_title("Heterogeneity Versus SRM+LLM Encoding Strength")
ax.set_xlabel("Within-network heterogeneity index")
ax.set_ylabel("Mean best SRM+LLM encoding r")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
if HAS_SEABORN:
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(FIG_DIR / "step11E_heterogeneity_vs_encoding_strength.png", dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5.8))
fam_plot = family_heterogeneity_df.sort_values("heterogeneity_index", ascending=True)
ax.barh(fam_plot["network_family"], fam_plot["heterogeneity_index"], color="#1B9E77", alpha=0.85)
ax.set_title("Network-Family Layer-Depth Heterogeneity")
ax.set_xlabel("Heterogeneity index")
ax.set_ylabel("Network family")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
plt.tight_layout()
fig.savefig(FIG_DIR / "step11E_family_heterogeneity_index_barplot.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# =========================
# Summary JSON and validation
# =========================

summary = {
    "step": "step11E_within_network_heterogeneity",
    "analysis_goal": "Quantify whether parcel-level SRM+LLM preferred layer depth is consistent or heterogeneous within each functional network.",
    "depth_bin_definition": {
        "early": "log depth < 50%",
        "early_middle": "50% <= log depth < 72%",
        "middle_late": "72% <= log depth < 90%",
        "late": "log depth >= 90%",
    },
    "n_parcels": int(parcel_df["parcel_id"].nunique()),
    "n_networks": int(parcel_df["roi_name"].nunique()),
    "heterogeneity_definition": {
        "sd_preferred_depth": "Standard deviation of parcel best SRM layer depth (%) within a network.",
        "depth_bin_entropy": "Normalized entropy of early/early-middle/middle-late/late parcel proportions within a network.",
        "heterogeneity_index": "Average of sd_preferred_depth / 50 and four-bin depth_bin_entropy; larger values indicate broader within-network layer-depth heterogeneity.",
    },
    "top_heterogeneous_networks": top_heterogeneous_networks_df.head(10).to_dict(orient="records"),
    "top_consistent_networks": top_consistent_networks_df.head(10).to_dict(orient="records"),
    "outputs": {
        "csv_dir": str(CSV_DIR),
        "xlsx_path": str(xlsx_path),
        "figure_dir": str(FIG_DIR),
    },
    "elapsed_minutes": float((time.time() - start_time) / 60),
}

summary_path = JSON_DIR / "step11E_within_network_heterogeneity_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11E validation")
print(f"network_heterogeneity_df shape: {network_heterogeneity_df.shape}")
print(f"family_heterogeneity_df shape : {family_heterogeneity_df.shape}")
print(f"parcel_with_network_df shape  : {parcel_with_network_df.shape}")
print(f"Summary JSON                  : {summary_path}")

show_cols = [
    "roi_name", "network_family", "n_parcels",
    "mean_preferred_depth", "sd_preferred_depth",
    "early_parcel_proportion", "early_middle_parcel_proportion", "middle_late_parcel_proportion", "late_parcel_proportion",
    "dominant_depth_bin_label", "dominant_depth_bin_fraction",
    "depth_bin_entropy", "heterogeneity_index",
    "mean_best_srm_r", "mean_same_layer_gain",
]

print("\nMost heterogeneous networks:")
print(network_heterogeneity_df[show_cols].head(12).to_string(index=False))

print("\nMost internally consistent networks:")
print(top_consistent_networks_df[show_cols].head(12).to_string(index=False))

display(network_heterogeneity_df[show_cols].head(12))
display(top_consistent_networks_df[show_cols].head(12))

print(f"\nTotal elapsed: {(time.time() - start_time) / 60:.2f} min")
print("Step 11E completed successfully.")


In [ ]:

# ============================================================================
# Step 11F: Network-by-Network Spatial Gradient Analysis
# ============================================================================
# This step tests spatial gradients within each functional network separately.
# The key question is not whether the whole cortex has one global gradient, but
# whether each Schaefer network/subregion shows its own parcel-level spatial
# organization in preferred SRM+LLM layer depth.
#
# For every network, we test whether parcel-level preferred layer depth varies
# along signed left-right, medial-lateral distance, posterior-anterior, and
# inferior-superior axes. The primary analysis is single-axis correlation; a
# fixed full spatial model is kept as a secondary check. This avoids broad
# model-shopping across many alternative spatial models.

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

try:
    from scipy import stats
except Exception as exc:
    raise ImportError("Step 11F requires scipy for correlations and OLS model tests.") from exc

start_time = time.time()

pd.set_option("display.float_format", lambda x: f"{x:.5f}")
np.set_printoptions(precision=5, suppress=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
OUTPUT_ROOT = BASE_DIR / "parcel_layer_network_outputs"
STEP11D_DIR = OUTPUT_ROOT / "step11D_parcel_level_srm_gain_maps"
STEP11F_DIR = OUTPUT_ROOT / "step11F_spatial_gradient_analysis"
CSV_DIR = STEP11F_DIR / "csv"
FIG_DIR = STEP11F_DIR / "figures"
JSON_DIR = STEP11F_DIR / "json"
XLSX_DIR = STEP11F_DIR / "xlsx"

for d in [STEP11F_DIR, CSV_DIR, FIG_DIR, JSON_DIR, XLSX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PARCEL_GAIN_PATH = STEP11D_DIR / "csv" / "step11D_parcel_srm_gain_metrics.csv"
if not PARCEL_GAIN_PATH.exists():
    raise FileNotFoundError(f"Missing Step 11D parcel table: {PARCEL_GAIN_PATH}")

parcel_df = pd.read_csv(PARCEL_GAIN_PATH)

required_cols = [
    "parcel_id", "parcel_label", "roi_name", "network_family",
    "best_srm_log_depth_percent", "srm_depth_center_of_mass",
    "best_srm_r", "same_layer_gain", "contextual_minus_early_gain",
    "srm_preferred_layer_confidence", "srm_preferred_layer_is_interpretable",
    "centroid_x", "centroid_y", "centroid_z",
]
missing_cols = [c for c in required_cols if c not in parcel_df.columns]
if missing_cols:
    raise ValueError(f"Step 11D parcel table is missing required columns: {missing_cols}")

# Keep confidence as a categorical label. Do not coerce it into NaN.
def parse_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)) and np.isfinite(x):
        return bool(int(x))
    if isinstance(x, str):
        return x.strip().lower() in {"true", "1", "yes", "y"}
    return False

parcel_df["srm_preferred_layer_is_interpretable"] = parcel_df["srm_preferred_layer_is_interpretable"].apply(parse_bool)

numeric_cols = [
    "best_srm_log_depth_percent", "srm_depth_center_of_mass",
    "best_srm_r", "same_layer_gain", "contextual_minus_early_gain",
    "centroid_x", "centroid_y", "centroid_z",
]
for col in numeric_cols:
    parcel_df[col] = pd.to_numeric(parcel_df[col], errors="coerce")

confidence_map = {"low": 1, "moderate": 2, "high": 3}
parcel_df["confidence_numeric"] = (
    parcel_df["srm_preferred_layer_confidence"].astype(str).str.strip().str.lower().map(confidence_map)
)
parcel_df["abs_centroid_x"] = parcel_df["centroid_x"].abs()

critical_numeric_cols = [
    "best_srm_log_depth_percent", "best_srm_r", "same_layer_gain", "contextual_minus_early_gain",
    "centroid_x", "abs_centroid_x", "centroid_y", "centroid_z",
]
if parcel_df[critical_numeric_cols].isna().any().any():
    bad = parcel_df[parcel_df[critical_numeric_cols].isna().any(axis=1)].head(10)
    raise ValueError(f"Non-finite values detected in Step 11F critical inputs. First rows:\n{bad}")

NETWORK_ORDER = [
    "VisCent", "VisPeri", "SomMotA", "SomMotB", "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB", "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC", "DefaultA", "DefaultB", "DefaultC", "TempPar",
]

TARGET_SPECS = [
    {
        "target": "best_srm_log_depth_percent",
        "label": "Best SRM layer depth (%)",
        "short_label": "preferred_depth",
        "kind": "depth",
    },
]

AXIS_SPECS = [
    ("centroid_x", "signed left-right x", "MNI x; negative = left, positive = right"),
    ("abs_centroid_x", "medial-lateral |x|", "Absolute MNI x; larger = farther from midline"),
    ("centroid_y", "posterior-anterior y", "MNI y; negative = posterior, positive = anterior"),
    ("centroid_z", "inferior-superior z", "MNI z; lower = inferior, higher = superior"),
]
AXIS_LABEL_MAP = {axis: name for axis, name, _ in AXIS_SPECS}

MODEL_SPECS = [
    {"model_name": "signed_x_only", "predictors": ["centroid_x"], "model_family": "primary_single_axis"},
    {"model_name": "medial_lateral_abs_x_only", "predictors": ["abs_centroid_x"], "model_family": "primary_single_axis"},
    {"model_name": "posterior_anterior_y_only", "predictors": ["centroid_y"], "model_family": "primary_single_axis"},
    {"model_name": "inferior_superior_z_only", "predictors": ["centroid_z"], "model_family": "primary_single_axis"},
    {"model_name": "fixed_full_signed_x_abs_x_y_z", "predictors": ["centroid_x", "abs_centroid_x", "centroid_y", "centroid_z"], "model_family": "secondary_fixed_full_model"},
]

MIN_PARCELS_FOR_AXIS = 12
MIN_PARCELS_FOR_MULTIAXIS = 12
HIGH_PERFORMING_R_THRESHOLD = 0.05

high_performing_df = parcel_df[
    (parcel_df["best_srm_r"] >= HIGH_PERFORMING_R_THRESHOLD) &
    (parcel_df["same_layer_gain"] > 0)
].copy()

analysis_sets = [
    ("all_parcels", parcel_df.copy()),
    ("interpretable_only", parcel_df[parcel_df["srm_preferred_layer_is_interpretable"]].copy()),
    ("high_performing", high_performing_df),
]

# =========================
# Helper functions
# =========================

def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"

def fdr_bh(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adj = ranked * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    temp = np.empty_like(adj)
    temp[order] = adj
    out[valid] = temp
    return out

def standardize_vector(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full_like(x, np.nan, dtype=float)
    return (x - mu) / sd

def fit_ols_model(df, target_col, predictor_cols):
    cols = list(predictor_cols) + [target_col]
    d = df[cols].dropna().copy()
    n = len(d)
    p = len(predictor_cols)
    if n <= p + 2:
        return None

    y = standardize_vector(d[target_col].astype(float).values)
    X_cols = [standardize_vector(d[col].astype(float).values) for col in predictor_cols]
    X_no_intercept = np.column_stack(X_cols)
    valid = np.isfinite(y) & np.isfinite(X_no_intercept).all(axis=1)
    y = y[valid]
    X_no_intercept = X_no_intercept[valid]
    n = len(y)
    if n <= p + 2:
        return None

    X = np.column_stack([np.ones(n), X_no_intercept])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    resid = y - y_hat
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot
    adj_r2 = np.nan if n <= p + 1 or not np.isfinite(r2) else 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1)

    if np.isfinite(r2) and r2 < 1 and n > p + 1:
        f_value = (r2 / p) / ((1 - r2) / (n - p - 1))
        model_p = float(stats.f.sf(f_value, p, n - p - 1))
    else:
        f_value = np.nan
        model_p = np.nan

    dof = n - p - 1
    coef = {"intercept": float(beta[0])}
    tvals = {}
    pvals = {}
    try:
        mse = ss_res / dof if dof > 0 else np.nan
        cov_beta = mse * np.linalg.inv(X.T @ X)
        se = np.sqrt(np.diag(cov_beta))
        t_all = beta / se
        p_all = 2 * stats.t.sf(np.abs(t_all), dof)
    except Exception:
        t_all = np.full_like(beta, np.nan, dtype=float)
        p_all = np.full_like(beta, np.nan, dtype=float)

    for j, predictor in enumerate(predictor_cols, start=1):
        coef[f"beta_{predictor}"] = float(beta[j])
        tvals[f"t_{predictor}"] = float(t_all[j])
        pvals[f"p_{predictor}"] = float(p_all[j])

    return {
        "n_parcels": int(n),
        "n_predictors": int(p),
        "r2": float(r2),
        "adj_r2": float(adj_r2),
        "f_value": float(f_value),
        "model_p": float(model_p),
        **coef,
        **tvals,
        **pvals,
    }

def axis_correlation(df, target_col, axis_col):
    d = df[[target_col, axis_col]].dropna().copy()
    if len(d) < MIN_PARCELS_FOR_AXIS:
        return None
    x = d[axis_col].astype(float).values
    y = d[target_col].astype(float).values
    if np.nanstd(x, ddof=1) <= 0 or np.nanstd(y, ddof=1) <= 0:
        return None
    r, p = stats.pearsonr(x, y)
    return {"n_parcels": int(len(d)), "r": float(r), "p_value": float(p)}

# =========================
# 1. Network-by-network models
# =========================

axis_rows = []
model_rows = []
network_input_rows = []

for analysis_set, df_set in analysis_sets:
    for (roi_name, network_family), network_df in df_set.groupby(["roi_name", "network_family"], observed=True):
        network_input_rows.append({
            "analysis_set": analysis_set,
            "roi_name": roi_name,
            "network_family": network_family,
            "n_parcels_available": int(len(network_df)),
            "mean_best_srm_r": float(network_df["best_srm_r"].mean()) if len(network_df) else np.nan,
            "mean_same_layer_gain": float(network_df["same_layer_gain"].mean()) if len(network_df) else np.nan,
            "mean_preferred_depth": float(network_df["best_srm_log_depth_percent"].mean()) if len(network_df) else np.nan,
        })

        for target_spec in TARGET_SPECS:
            target_col = target_spec["target"]

            for axis_col, axis_name, axis_definition in AXIS_SPECS:
                corr = axis_correlation(network_df, target_col, axis_col)
                if corr is None:
                    continue
                axis_rows.append({
                    "analysis_set": analysis_set,
                    "roi_name": roi_name,
                    "network_family": network_family,
                    "target": target_col,
                    "target_label": target_spec["label"],
                    "target_kind": target_spec["kind"],
                    "axis": axis_col,
                    "axis_name": axis_name,
                    "axis_definition": axis_definition,
                    **corr,
                })

            for model_spec in MODEL_SPECS:
                predictors = model_spec["predictors"]
                min_n = MIN_PARCELS_FOR_AXIS if model_spec["model_family"] == "primary_single_axis" else MIN_PARCELS_FOR_MULTIAXIS
                if len(network_df) < min_n:
                    continue
                model_result = fit_ols_model(network_df, target_col, predictors)
                if model_result is None:
                    continue
                model_rows.append({
                    "analysis_set": analysis_set,
                    "roi_name": roi_name,
                    "network_family": network_family,
                    "target": target_col,
                    "target_label": target_spec["label"],
                    "target_kind": target_spec["kind"],
                    "model_name": model_spec["model_name"],
                    "model_family": model_spec["model_family"],
                    "predictors": "+".join(predictors),
                    **model_result,
                })

axis_corr_df = pd.DataFrame(axis_rows)
spatial_model_df = pd.DataFrame(model_rows)
network_input_df = pd.DataFrame(network_input_rows)

if len(axis_corr_df):
    axis_corr_df["p_fdr_within_target_axis_tests"] = np.nan
    for (analysis_set, target), idx in axis_corr_df.groupby(["analysis_set", "target"]).groups.items():
        axis_corr_df.loc[idx, "p_fdr_within_target_axis_tests"] = fdr_bh(axis_corr_df.loc[idx, "p_value"].values)
    axis_corr_df = axis_corr_df.sort_values(
        ["analysis_set", "target", "p_fdr_within_target_axis_tests", "p_value", "roi_name"],
        ascending=[True, True, True, True, True],
    ).reset_index(drop=True)

if len(spatial_model_df):
    spatial_model_df["model_p_fdr_within_target_models"] = np.nan
    for (analysis_set, target), idx in spatial_model_df.groupby(["analysis_set", "target"]).groups.items():
        spatial_model_df.loc[idx, "model_p_fdr_within_target_models"] = fdr_bh(spatial_model_df.loc[idx, "model_p"].values)
    spatial_model_df = spatial_model_df.sort_values(
        ["analysis_set", "target", "model_p_fdr_within_target_models", "adj_r2", "roi_name"],
        ascending=[True, True, True, False, True],
    ).reset_index(drop=True)

# Best model per network and target. This is the most useful table for interpretation.
best_rows = []
if len(spatial_model_df):
    for (analysis_set, roi_name, target), sub_df in spatial_model_df.groupby(["analysis_set", "roi_name", "target"], observed=True):
        sub_df = sub_df.copy().sort_values(["model_p_fdr_within_target_models", "model_p", "adj_r2"], ascending=[True, True, False])
        best_rows.append(sub_df.iloc[0].to_dict())
best_model_df = pd.DataFrame(best_rows)
if len(best_model_df):
    best_model_df = best_model_df.sort_values(
        ["analysis_set", "target", "model_p_fdr_within_target_models", "adj_r2"],
        ascending=[True, True, True, False],
    ).reset_index(drop=True)

# Best single axis per network and target, kept separate from multivariate models.
best_axis_rows = []
if len(axis_corr_df):
    for (analysis_set, roi_name, target), sub_df in axis_corr_df.groupby(["analysis_set", "roi_name", "target"], observed=True):
        sub_df = sub_df.copy()
        sub_df["abs_r"] = sub_df["r"].abs()
        sub_df = sub_df.sort_values(["p_fdr_within_target_axis_tests", "p_value", "abs_r"], ascending=[True, True, False])
        best_axis_rows.append(sub_df.iloc[0].to_dict())
best_axis_df = pd.DataFrame(best_axis_rows)
if len(best_axis_df):
    best_axis_df = best_axis_df.sort_values(
        ["analysis_set", "target", "p_fdr_within_target_axis_tests", "p_value"],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)

# Add network strength columns for interpretation.
network_strength_df = (
    parcel_df.groupby(["roi_name", "network_family"], as_index=False)
    .agg(
        mean_best_srm_r=("best_srm_r", "mean"),
        mean_same_layer_gain=("same_layer_gain", "mean"),
        mean_contextual_minus_early_gain=("contextual_minus_early_gain", "mean"),
        n_total_parcels=("parcel_id", "count"),
    )
)

for df_name in ["spatial_model_df", "best_model_df", "best_axis_df"]:
    df_obj = globals()[df_name]
    if len(df_obj):
        globals()[df_name] = df_obj.merge(network_strength_df, on=["roi_name", "network_family"], how="left")

# =========================
# 2. Save tables
# =========================

axis_corr_csv = CSV_DIR / "step11F_network_axiswise_spatial_correlations.csv"
spatial_model_csv = CSV_DIR / "step11F_network_spatial_model_comparison.csv"
best_model_csv = CSV_DIR / "step11F_network_best_spatial_gradient_models.csv"
best_axis_csv = CSV_DIR / "step11F_network_best_single_axis_gradients.csv"
network_input_csv = CSV_DIR / "step11F_network_analysis_input_counts.csv"
parcel_input_csv = CSV_DIR / "step11F_parcel_centroid_gradient_input_table.csv"
network_strength_csv = CSV_DIR / "step11F_network_encoding_strength_summary.csv"

axis_corr_df.to_csv(axis_corr_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
spatial_model_df.to_csv(spatial_model_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
best_model_df.to_csv(best_model_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
best_axis_df.to_csv(best_axis_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
network_input_df.to_csv(network_input_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
parcel_df.to_csv(parcel_input_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
network_strength_df.to_csv(network_strength_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

try:
    with pd.ExcelWriter(XLSX_DIR / "step11F_network_by_network_spatial_gradient_tables.xlsx", engine="openpyxl") as writer:
        network_input_df.to_excel(writer, sheet_name="network_input_counts", index=False)
        axis_corr_df.to_excel(writer, sheet_name="axis_correlations", index=False)
        spatial_model_df.to_excel(writer, sheet_name="model_comparison", index=False)
        best_model_df.to_excel(writer, sheet_name="best_models", index=False)
        best_axis_df.to_excel(writer, sheet_name="best_single_axes", index=False)
        network_strength_df.to_excel(writer, sheet_name="network_strength", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")

# =========================
# 3. Figures
# =========================

preferred_best_model_df = best_model_df[
    (best_model_df["analysis_set"] == "high_performing") &
    (best_model_df["target"] == "best_srm_log_depth_percent")
].copy()
if len(preferred_best_model_df) == 0:
    preferred_best_model_df = best_model_df[
        (best_model_df["analysis_set"] == "interpretable_only") &
        (best_model_df["target"] == "best_srm_log_depth_percent")
    ].copy()

if len(preferred_best_model_df):
    preferred_best_model_df["roi_name"] = pd.Categorical(preferred_best_model_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
    fig, ax = plt.subplots(figsize=(12, 7))
    plot_df = preferred_best_model_df.sort_values("adj_r2", ascending=False)
    sns.barplot(data=plot_df, y="roi_name", x="adj_r2", hue="model_family", dodge=False, ax=ax)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_title("Best Within-Network Spatial Gradient Model")
    ax.set_xlabel("Adjusted R² for preferred layer depth")
    ax.set_ylabel("Network")
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.legend(title="Model type", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "step11F_best_network_spatial_gradient_models.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

preferred_axis_df = best_axis_df[
    (best_axis_df["analysis_set"] == "high_performing") &
    (best_axis_df["target"] == "best_srm_log_depth_percent")
].copy()
if len(preferred_axis_df) == 0:
    preferred_axis_df = best_axis_df[
        (best_axis_df["analysis_set"] == "interpretable_only") &
        (best_axis_df["target"] == "best_srm_log_depth_percent")
    ].copy()

if len(preferred_axis_df):
    preferred_axis_df["roi_name"] = pd.Categorical(preferred_axis_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
    fig, ax = plt.subplots(figsize=(12, 7))
    plot_df = preferred_axis_df.sort_values("r", ascending=True)
    sns.barplot(data=plot_df, y="roi_name", x="r", hue="axis_name", dodge=False, ax=ax)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_title("Best Single-Axis Spatial Gradient By Network")
    ax.set_xlabel("Pearson r with preferred layer depth")
    ax.set_ylabel("Network")
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.legend(title="Best axis", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "step11F_best_network_single_axis_gradients.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# Heatmap: network x model adjusted R2 for high-performing preferred-depth models.
heat_df = spatial_model_df[
    (spatial_model_df["analysis_set"] == "high_performing") &
    (spatial_model_df["target"] == "best_srm_log_depth_percent")
].copy()
if len(heat_df):
    pivot = heat_df.pivot_table(index="roi_name", columns="model_name", values="adj_r2", aggfunc="max")
    pivot = pivot.reindex([n for n in NETWORK_ORDER if n in pivot.index])
    fig, ax = plt.subplots(figsize=(13, 8))
    sns.heatmap(pivot, cmap="rocket_r", annot=True, fmt=".2f", linewidths=0.4, linecolor="white", cbar_kws={"label": "Adjusted R²"}, ax=ax)
    ax.set_title("Network-Specific Spatial Model Comparison")
    ax.set_xlabel("Spatial model")
    ax.set_ylabel("Network")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "step11F_network_spatial_model_comparison_heatmap.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# Scatter panels: one panel per network using its best single axis.
scatter_source_df = parcel_df.copy()
fig_networks = [n for n in NETWORK_ORDER if n in set(preferred_axis_df["roi_name"])]
if len(fig_networks):
    ncols = 4
    nrows = int(np.ceil(len(fig_networks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.8 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")

    for ax, roi_name in zip(axes.ravel(), fig_networks):
        row = preferred_axis_df[preferred_axis_df["roi_name"] == roi_name].iloc[0]
        axis_col = row["axis"]
        sub_df = scatter_source_df[scatter_source_df["roi_name"] == roi_name].copy()
        if row["analysis_set"] == "high_performing":
            sub_df = sub_df[(sub_df["best_srm_r"] >= HIGH_PERFORMING_R_THRESHOLD) & (sub_df["same_layer_gain"] > 0)].copy()
        elif row["analysis_set"] == "interpretable_only":
            sub_df = sub_df[sub_df["srm_preferred_layer_is_interpretable"]].copy()
        ax.axis("on")
        sns.regplot(
            data=sub_df,
            x=axis_col,
            y="best_srm_log_depth_percent",
            scatter_kws={"s": 32, "alpha": 0.75, "edgecolors": "white", "linewidths": 0.4},
            line_kws={"color": "#D95F02", "linewidth": 2.0},
            ax=ax,
        )
        ax.set_title(f"{roi_name}: {AXIS_LABEL_MAP.get(axis_col, axis_col)}\nr={row['r']:.2f}, FDR p={row['p_fdr_within_target_axis_tests']:.3f}", fontsize=10)
        ax.set_xlabel(axis_col)
        ax.set_ylabel("Layer depth (%)")
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))

    fig.suptitle("Network-by-Network Best Single-Axis Gradient", y=1.01, fontsize=16)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "step11F_network_by_network_best_axis_scatter_grid.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# Scatter: network strength versus best spatial gradient strength.
if len(preferred_best_model_df):
    fig, ax = plt.subplots(figsize=(8.5, 6.5))
    sns.scatterplot(
        data=preferred_best_model_df,
        x="mean_best_srm_r",
        y="adj_r2",
        hue="network_family",
        style="model_family",
        s=105,
        edgecolor="white",
        linewidth=0.8,
        ax=ax,
    )
    if preferred_best_model_df["mean_best_srm_r"].notna().sum() >= 4:
        sns.regplot(data=preferred_best_model_df, x="mean_best_srm_r", y="adj_r2", scatter=False, color="#333333", ax=ax)
    for _, row in preferred_best_model_df.iterrows():
        if np.isfinite(row["mean_best_srm_r"]) and np.isfinite(row["adj_r2"]):
            ax.text(row["mean_best_srm_r"], row["adj_r2"], row["roi_name"], fontsize=8, ha="left", va="bottom")
    ax.set_title("Encoding Strength And Within-Network Spatial Organization")
    ax.set_xlabel("Network mean best SRM+LLM encoding r")
    ax.set_ylabel("Best within-network spatial model adjusted R²")
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.legend(title="Network family / model type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "step11F_encoding_strength_vs_network_spatial_gradient.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# =========================
# 4. Summary
# =========================

sig_best_models = best_model_df[
    (best_model_df["target"] == "best_srm_log_depth_percent") &
    (best_model_df["model_p_fdr_within_target_models"] < 0.05)
].copy() if len(best_model_df) else pd.DataFrame()

sig_best_axes = best_axis_df[
    (best_axis_df["target"] == "best_srm_log_depth_percent") &
    (best_axis_df["p_fdr_within_target_axis_tests"] < 0.05)
].copy() if len(best_axis_df) else pd.DataFrame()

total_elapsed = (time.time() - start_time) / 60.0
summary = {
    "step": "step11F_network_by_network_spatial_gradient_analysis",
    "analysis_goal": "Test preferred-layer spatial gradients separately inside each functional network/subregion using primary single-axis models and one fixed full spatial model.",
    "primary_target": "best_srm_log_depth_percent",
    "secondary_targets": [],
    "analysis_sets": [name for name, _ in analysis_sets],
    "axis_definitions": {
        "centroid_x": "signed left-right MNI coordinate; negative is left hemisphere, positive is right hemisphere",
        "abs_centroid_x": "absolute MNI x coordinate; medial-lateral distance from midline",
        "centroid_y": "posterior-anterior MNI coordinate; more positive is more anterior",
        "centroid_z": "inferior-superior MNI coordinate; more positive is more superior",
    },
    "model_comparison": [
        {"model_name": spec["model_name"], "predictors": spec["predictors"], "model_family": spec["model_family"]}
        for spec in MODEL_SPECS
    ],
    "min_parcels_for_single_axis": int(MIN_PARCELS_FOR_AXIS),
    "min_parcels_for_multiaxis": int(MIN_PARCELS_FOR_MULTIAXIS),
    "high_performing_definition": f"best_srm_r >= {HIGH_PERFORMING_R_THRESHOLD:.2f} and same_layer_gain > 0",
    "n_parcels_total": int(len(parcel_df)),
    "n_interpretable_parcels": int(parcel_df["srm_preferred_layer_is_interpretable"].sum()),
    "n_high_performing_parcels": int(len(high_performing_df)),
    "n_axis_tests": int(len(axis_corr_df)),
    "n_spatial_models": int(len(spatial_model_df)),
    "n_significant_best_models_for_preferred_depth": int(len(sig_best_models)),
    "n_significant_best_axes_for_preferred_depth": int(len(sig_best_axes)),
    "outputs": {
        "axis_correlations_csv": str(axis_corr_csv),
        "spatial_model_comparison_csv": str(spatial_model_csv),
        "best_model_csv": str(best_model_csv),
        "best_axis_csv": str(best_axis_csv),
        "figure_dir": str(FIG_DIR),
    },
    "elapsed_minutes": float(total_elapsed),
}

with open(JSON_DIR / "step11F_network_by_network_spatial_gradient_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11F completed: Network-by-network spatial gradient analysis")
print(f"Total parcels: {len(parcel_df)}")
print(f"High-performing parcels: {len(high_performing_df)}")
print(f"Axis-wise tests: {len(axis_corr_df)}")
print(f"Spatial models: {len(spatial_model_df)}")
print(f"Saved tables to: {CSV_DIR}")
print(f"Saved figures to: {FIG_DIR}")
print(f"Elapsed: {total_elapsed:.2f} min")

if len(best_model_df):
    print("\nBest network-specific spatial models for preferred layer depth:")
    display_cols = [
        "analysis_set", "roi_name", "network_family", "model_name", "model_family",
        "n_parcels", "adj_r2", "model_p", "model_p_fdr_within_target_models",
        "mean_best_srm_r", "mean_same_layer_gain",
    ]
    print(
        best_model_df[best_model_df["target"] == "best_srm_log_depth_percent"][display_cols]
        .head(30)
        .to_string(index=False)
    )

if len(best_axis_df):
    print("\nBest single-axis gradients for preferred layer depth:")
    display_cols = [
        "analysis_set", "roi_name", "network_family", "axis_name", "n_parcels",
        "r", "p_value", "p_fdr_within_target_axis_tests",
    ]
    print(
        best_axis_df[best_axis_df["target"] == "best_srm_log_depth_percent"][display_cols]
        .head(30)
        .to_string(index=False)
    )


In [ ]:

# ============================================================================
# Step 11G: Final Parcel-Level Summary Figures And Tables
# ============================================================================
# This final parcel-level step integrates the main atlas-based outputs from
# Steps 11D-11F. It creates publication-facing summary tables, statistical
# follow-up tests, and compact figures showing network-level layer preference,
# SRM gain, within-network heterogeneity, and exploratory spatial gradients.
#
# This cell does not refit any encoding model. It only summarizes parcel-level
# results that were already generated in the previous atlas-analysis steps.

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

try:
    from scipy import stats
except Exception as exc:
    raise ImportError("Step 11G requires scipy for statistical follow-up tests.") from exc

start_time = time.time()

pd.set_option("display.float_format", lambda x: f"{x:.5f}")
np.set_printoptions(precision=5, suppress=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
OUTPUT_ROOT = BASE_DIR / "parcel_layer_network_outputs"
STEP11D_DIR = OUTPUT_ROOT / "step11D_parcel_level_srm_gain_maps"
STEP11E_DIR = OUTPUT_ROOT / "step11E_within_network_heterogeneity"
STEP11F_DIR = OUTPUT_ROOT / "step11F_spatial_gradient_analysis"
STEP11G_DIR = OUTPUT_ROOT / "step11G_final_parcel_level_summary"
CSV_DIR = STEP11G_DIR / "csv"
FIG_DIR = STEP11G_DIR / "figures"
JSON_DIR = STEP11G_DIR / "json"
XLSX_DIR = STEP11G_DIR / "xlsx"

for d in [STEP11G_DIR, CSV_DIR, FIG_DIR, JSON_DIR, XLSX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PARCEL_GAIN_PATH = STEP11D_DIR / "csv" / "step11D_parcel_srm_gain_metrics.csv"
NETWORK_GAIN_PATH = STEP11D_DIR / "csv" / "step11D_network_srm_gain_summary.csv"
NETWORK_HETEROGENEITY_PATH = STEP11E_DIR / "csv" / "step11E_network_within_network_heterogeneity_summary.csv"
FAMILY_HETEROGENEITY_PATH = STEP11E_DIR / "csv" / "step11E_family_within_network_heterogeneity_summary.csv"
BEST_GRADIENT_MODEL_PATH = STEP11F_DIR / "csv" / "step11F_network_best_spatial_gradient_models.csv"
BEST_AXIS_PATH = STEP11F_DIR / "csv" / "step11F_network_best_single_axis_gradients.csv"

required_paths = [
    PARCEL_GAIN_PATH,
    NETWORK_GAIN_PATH,
    NETWORK_HETEROGENEITY_PATH,
    FAMILY_HETEROGENEITY_PATH,
    BEST_GRADIENT_MODEL_PATH,
    BEST_AXIS_PATH,
]
for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing required Step 11 input: {p}")

parcel_df = pd.read_csv(PARCEL_GAIN_PATH)
network_gain_df = pd.read_csv(NETWORK_GAIN_PATH)
network_heterogeneity_df = pd.read_csv(NETWORK_HETEROGENEITY_PATH)
family_heterogeneity_df = pd.read_csv(FAMILY_HETEROGENEITY_PATH)
best_gradient_model_df = pd.read_csv(BEST_GRADIENT_MODEL_PATH)
best_axis_df = pd.read_csv(BEST_AXIS_PATH)

NETWORK_ORDER = [
    "VisCent", "VisPeri", "SomMotA", "SomMotB", "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB", "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC", "DefaultA", "DefaultB", "DefaultC", "TempPar",
]
FAMILY_ORDER = ["Visual", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Control", "Default", "TempPar"]
FAMILY_COLORS = {
    "Visual": "#E69F00",
    "SomMot": "#56B4E9",
    "DorsAttn": "#6A3D9A",
    "SalVentAttn": "#B2ABD2",
    "Control": "#2CA25F",
    "Default": "#1B7837",
    "Limbic": "#DD3497",
    "TempPar": "#E7298A",
}
DEPTH_BIN_ORDER = ["early", "early_middle", "middle_late", "late"]
DEPTH_BIN_LABELS = {
    "early": "Early (<50%)",
    "early_middle": "Early-middle (50-72%)",
    "middle_late": "Middle-late (72-90%)",
    "late": "Late (>=90%)",
}
DEPTH_BIN_COLORS = {
    "early": "#8DA0CB",
    "early_middle": "#66C2A5",
    "middle_late": "#FC8D62",
    "late": "#D95F02",
}

# =========================
# Helper functions
# =========================

def parse_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)) and np.isfinite(x):
        return bool(int(x))
    if isinstance(x, str):
        return x.strip().lower() in {"true", "1", "yes", "y"}
    return False

def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"

def format_p5(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.00001:
        return "<0.00001"
    return f"{float(p):.5f}"

def fdr_bh(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adj = ranked * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    temp = np.empty_like(adj)
    temp[order] = adj
    out[valid] = temp
    return out

def safe_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def add_sig_stars(p):
    if p is None or not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def savefig_clean(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

# =========================
# 1. Clean and merge final tables
# =========================

parcel_df["srm_preferred_layer_is_interpretable"] = parcel_df["srm_preferred_layer_is_interpretable"].apply(parse_bool)
parcel_df = safe_numeric(parcel_df, [
    "best_srm_log_depth_percent", "best_srm_r", "same_layer_gain", "max_layer_gain",
    "contextual_minus_early_gain", "late_minus_early_gain", "srm_depth_center_of_mass",
    "srm_preferred_layer_confidence", "centroid_x", "centroid_y", "centroid_z",
])

# The confidence column is usually categorical; keep it categorical if numeric conversion failed.
if "srm_preferred_layer_confidence" in parcel_df.columns and parcel_df["srm_preferred_layer_confidence"].isna().all():
    original_conf = pd.read_csv(PARCEL_GAIN_PATH, usecols=["parcel_id", "srm_preferred_layer_confidence"])
    parcel_df = parcel_df.drop(columns=["srm_preferred_layer_confidence"]).merge(original_conf, on="parcel_id", how="left")

# Final network table starts from Step 11D and adds heterogeneity and spatial-gradient summaries.
network_final_df = network_gain_df.copy()
if "roi_name" not in network_final_df.columns and "network" in network_final_df.columns:
    network_final_df = network_final_df.rename(columns={"network": "roi_name"})

hetero_cols = [
    c for c in network_heterogeneity_df.columns
    if c in {
        "roi_name", "mean_preferred_depth", "sd_preferred_depth", "iqr_preferred_depth",
        "range_preferred_depth", "mean_depth_center_of_mass", "dominant_depth_bin",
        "dominant_depth_bin_label", "dominant_depth_bin_fraction", "depth_bin_entropy",
        "heterogeneity_index", "early_parcel_proportion", "early_middle_parcel_proportion",
        "middle_late_parcel_proportion", "late_parcel_proportion",
    }
]
network_final_df = network_final_df.merge(network_heterogeneity_df[hetero_cols], on="roi_name", how="left")

# Prefer high-performing spatial-gradient summaries per network, but fall back
# network-by-network to interpretable-only when high-performing results are absent.
grad_high = best_gradient_model_df[
    (best_gradient_model_df["analysis_set"] == "high_performing") &
    (best_gradient_model_df["target"] == "best_srm_log_depth_percent")
].copy()
grad_interp = best_gradient_model_df[
    (best_gradient_model_df["analysis_set"] == "interpretable_only") &
    (best_gradient_model_df["target"] == "best_srm_log_depth_percent")
].copy()
gradient_pref_df = pd.concat([grad_high, grad_interp], ignore_index=True)
if len(gradient_pref_df):
    gradient_pref_df["analysis_priority"] = gradient_pref_df["analysis_set"].map({
        "high_performing": 0,
        "interpretable_only": 1,
    }).fillna(9)
    gradient_pref_df = (
        gradient_pref_df
        .sort_values(["roi_name", "analysis_priority"])
        .drop_duplicates("roi_name", keep="first")
        .drop(columns=["analysis_priority"])
        .reset_index(drop=True)
    )

axis_high = best_axis_df[
    (best_axis_df["analysis_set"] == "high_performing") &
    (best_axis_df["target"] == "best_srm_log_depth_percent")
].copy()
axis_interp = best_axis_df[
    (best_axis_df["analysis_set"] == "interpretable_only") &
    (best_axis_df["target"] == "best_srm_log_depth_percent")
].copy()
axis_pref_df = pd.concat([axis_high, axis_interp], ignore_index=True)
if len(axis_pref_df):
    axis_pref_df["analysis_priority"] = axis_pref_df["analysis_set"].map({
        "high_performing": 0,
        "interpretable_only": 1,
    }).fillna(9)
    axis_pref_df = (
        axis_pref_df
        .sort_values(["roi_name", "analysis_priority"])
        .drop_duplicates("roi_name", keep="first")
        .drop(columns=["analysis_priority"])
        .reset_index(drop=True)
    )

gradient_cols = [
    "roi_name", "analysis_set", "model_name", "model_family", "predictors", "n_parcels",
    "adj_r2", "model_p", "model_p_fdr_within_target_models",
]
gradient_pref_df = gradient_pref_df[[c for c in gradient_cols if c in gradient_pref_df.columns]].rename(columns={
    "analysis_set": "spatial_model_analysis_set",
    "model_name": "best_spatial_model",
    "model_family": "best_spatial_model_family",
    "predictors": "best_spatial_model_predictors",
    "n_parcels": "spatial_model_n_parcels",
    "adj_r2": "best_spatial_model_adj_r2",
    "model_p": "best_spatial_model_p",
    "model_p_fdr_within_target_models": "best_spatial_model_fdr_p",
})
network_final_df = network_final_df.merge(gradient_pref_df, on="roi_name", how="left")

axis_cols = ["roi_name", "axis_name", "axis", "n_parcels", "r", "p_value", "p_fdr_within_target_axis_tests"]
axis_pref_df = axis_pref_df[[c for c in axis_cols if c in axis_pref_df.columns]].rename(columns={
    "axis_name": "best_single_axis_name",
    "axis": "best_single_axis",
    "n_parcels": "best_single_axis_n_parcels",
    "r": "best_single_axis_r",
    "p_value": "best_single_axis_p",
    "p_fdr_within_target_axis_tests": "best_single_axis_fdr_p",
})
network_final_df = network_final_df.merge(axis_pref_df, on="roi_name", how="left")

if network_final_df["roi_name"].duplicated().any():
    dup = network_final_df[network_final_df["roi_name"].duplicated(keep=False)]
    raise ValueError(f"Duplicate network rows after merge:\\n{dup[['roi_name', 'network_family']].head()}")

if network_final_df["roi_name"].nunique() != 17:
    raise ValueError(
        f"Expected 17 networks, got {network_final_df['roi_name'].nunique()}."
    )

# Add stable ordering.
network_final_df["roi_name"] = pd.Categorical(network_final_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
network_final_df = network_final_df.sort_values("roi_name").reset_index(drop=True)
parcel_df["roi_name"] = pd.Categorical(parcel_df["roi_name"], categories=NETWORK_ORDER, ordered=True)

# Final compact ranking tables.
rank_cols = [
    "roi_name", "network_family", "mean_best_srm_r", "mean_same_layer_gain",
    "mean_preferred_depth", "sd_preferred_depth", "heterogeneity_index",
    "best_single_axis_name", "best_single_axis_r", "best_single_axis_fdr_p",
    "best_spatial_model", "best_spatial_model_adj_r2", "best_spatial_model_fdr_p",
]
rank_cols = [c for c in rank_cols if c in network_final_df.columns]
network_ranking_df = network_final_df[rank_cols].copy()
if "mean_best_srm_r" in network_ranking_df.columns:
    network_ranking_df = network_ranking_df.sort_values("mean_best_srm_r", ascending=False).reset_index(drop=True)

# =========================
# 2. Statistical follow-up tests
# =========================

stats_rows = []

def add_kruskal_test(data_df, metric, group_col, label):
    if metric not in data_df.columns or group_col not in data_df.columns:
        return
    groups = []
    group_names = []
    for name, sub_df in data_df.groupby(group_col, observed=True):
        vals = pd.to_numeric(sub_df[metric], errors="coerce").dropna().values
        if len(vals) >= 2:
            groups.append(vals)
            group_names.append(str(name))
    if len(groups) < 2:
        return
    h_val, p_val = stats.kruskal(*groups)
    stats_rows.append({
        "analysis": label,
        "metric": metric,
        "grouping": group_col,
        "test": "Kruskal-Wallis rank-sum test",
        "n_groups": len(groups),
        "groups": "; ".join(group_names),
        "statistic": float(h_val),
        "p_value": float(p_val),
        "p_value_print": format_p5(p_val),
        "interpretation": "Tests whether the parcel-level metric differs across groups.",
    })

for metric in ["best_srm_log_depth_percent", "best_srm_r", "same_layer_gain", "contextual_minus_early_gain"]:
    add_kruskal_test(parcel_df, metric, "roi_name", "parcel_metric_differs_by_network")
    add_kruskal_test(parcel_df, metric, "network_family", "parcel_metric_differs_by_network_family")

# Correlations among major parcel-level quantities.
corr_metrics = [
    "best_srm_log_depth_percent", "best_srm_r", "same_layer_gain",
    "contextual_minus_early_gain", "srm_depth_center_of_mass",
]
for i, metric_a in enumerate(corr_metrics):
    for metric_b in corr_metrics[i + 1:]:
        if metric_a not in parcel_df.columns or metric_b not in parcel_df.columns:
            continue
        d = parcel_df[[metric_a, metric_b]].dropna()
        if len(d) < 5:
            continue
        rho, p_val = stats.spearmanr(d[metric_a].values, d[metric_b].values)
        stats_rows.append({
            "analysis": "parcel_metric_spearman_correlation",
            "metric": f"{metric_a} vs {metric_b}",
            "grouping": "all_parcels",
            "test": "Spearman correlation",
            "n_groups": 1,
            "groups": "all parcels",
            "n_observations": int(len(d)),
            "statistic": float(rho),
            "p_value": float(p_val),
            "p_value_print": format_p5(p_val),
            "interpretation": "Tests monotonic association between two parcel-level quantities.",
        })

# Network-level relationship between encoding strength and spatial organization.
if {"mean_best_srm_r", "best_spatial_model_adj_r2"}.issubset(network_final_df.columns):
    d = network_final_df[["mean_best_srm_r", "best_spatial_model_adj_r2"]].dropna()
    if len(d) >= 5:
        rho, p_val = stats.spearmanr(d["mean_best_srm_r"], d["best_spatial_model_adj_r2"])
        stats_rows.append({
            "analysis": "network_encoding_strength_vs_spatial_organization",
            "metric": "mean_best_srm_r vs best_spatial_model_adj_r2",
            "grouping": "network",
            "test": "Spearman correlation",
            "n_groups": 1,
            "groups": "networks",
            "n_observations": int(len(d)),
            "statistic": float(rho),
            "p_value": float(p_val),
            "p_value_print": format_p5(p_val),
            "interpretation": "Tests whether networks with stronger SRM+LLM encoding also show stronger within-network spatial organization.",
        })

stats_df = pd.DataFrame(stats_rows)
if len(stats_df):
    stats_df["p_fdr_all_step11G_tests"] = fdr_bh(stats_df["p_value"].values)
    stats_df["p_fdr_all_step11G_tests_print"] = stats_df["p_fdr_all_step11G_tests"].apply(format_p5)
    stats_df["significance_fdr"] = stats_df["p_fdr_all_step11G_tests"].apply(add_sig_stars)

# Concrete parcel hotspots are useful for manuscript interpretation, especially
# when a network-level effect is driven by a small number of high-performing parcels.
top_parcel_cols = [
    "parcel_id", "parcel_label", "roi_name", "network_family",
    "best_srm_layer_index", "best_srm_layer_label", "best_srm_log_depth_percent",
    "best_srm_depth_bin", "best_srm_r", "same_layer_gain",
    "contextual_minus_early_gain", "srm_preferred_layer_confidence",
    "centroid_x", "centroid_y", "centroid_z",
]
top_parcel_cols = [c for c in top_parcel_cols if c in parcel_df.columns]

top_parcels_by_r_df = (
    parcel_df[top_parcel_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["best_srm_r"])
    .sort_values("best_srm_r", ascending=False)
    .head(30)
    .reset_index(drop=True)
)

top_parcels_by_gain_df = (
    parcel_df[top_parcel_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["same_layer_gain"])
    .sort_values("same_layer_gain", ascending=False)
    .head(30)
    .reset_index(drop=True)
)

# =========================
# 3. Save final tables
# =========================

final_parcel_csv = CSV_DIR / "step11G_final_parcel_level_table.csv"
final_network_csv = CSV_DIR / "step11G_final_network_summary_table.csv"
network_ranking_csv = CSV_DIR / "step11G_network_ranking_summary.csv"
stats_csv = CSV_DIR / "step11G_statistical_followup_tests.csv"
top_parcels_by_r_csv = CSV_DIR / "step11G_top_parcels_by_best_srm_r.csv"
top_parcels_by_gain_csv = CSV_DIR / "step11G_top_parcels_by_same_layer_gain.csv"

parcel_df.to_csv(final_parcel_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
network_final_df.to_csv(final_network_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
network_ranking_df.to_csv(network_ranking_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
stats_df.to_csv(stats_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
top_parcels_by_r_df.to_csv(top_parcels_by_r_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
top_parcels_by_gain_df.to_csv(top_parcels_by_gain_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

try:
    with pd.ExcelWriter(XLSX_DIR / "step11G_final_parcel_level_summary_tables.xlsx", engine="openpyxl") as writer:
        network_ranking_df.to_excel(writer, sheet_name="network_ranking", index=False)
        network_final_df.to_excel(writer, sheet_name="network_summary", index=False)
        parcel_df.to_excel(writer, sheet_name="parcel_table", index=False)
        top_parcels_by_r_df.to_excel(writer, sheet_name="top_parcels_by_r", index=False)
        top_parcels_by_gain_df.to_excel(writer, sheet_name="top_parcels_by_gain", index=False)
        stats_df.to_excel(writer, sheet_name="statistical_tests", index=False)
        family_heterogeneity_df.to_excel(writer, sheet_name="family_summary", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")

# =========================
# 4. Final figures
# =========================

# Figure 1: network ranking by best SRM+LLM encoding and SRM gain.
plot_df = network_final_df.copy()
if "mean_best_srm_r" in plot_df.columns and "mean_same_layer_gain" in plot_df.columns:
    plot_df = plot_df.sort_values("mean_best_srm_r", ascending=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
    sns.barplot(data=plot_df, y="roi_name", x="mean_best_srm_r", hue="network_family", dodge=False, palette=FAMILY_COLORS, ax=axes[0])
    axes[0].set_title("Network Mean Best SRM+LLM Encoding")
    axes[0].set_xlabel("Mean best SRM+LLM encoding r")
    axes[0].set_ylabel("Network")
    axes[0].xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    axes[0].legend_.remove()

    sns.barplot(data=plot_df, y="roi_name", x="mean_same_layer_gain", hue="network_family", dodge=False, palette=FAMILY_COLORS, ax=axes[1])
    axes[1].set_title("Network Mean SRM Gain At Preferred Layer")
    axes[1].set_xlabel("Mean SRM gain")
    axes[1].set_ylabel("")
    axes[1].xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    handles, labels = axes[1].get_legend_handles_labels()
    axes[1].legend(handles, labels, title="Network family", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    savefig_clean(fig, FIG_DIR / "step11G_network_encoding_and_gain_summary.png")

# Figure 2: preferred layer depth distribution by network.
fig, ax = plt.subplots(figsize=(15, 7))
sns.stripplot(
    data=parcel_df,
    x="roi_name",
    y="best_srm_log_depth_percent",
    hue="network_family",
    palette=FAMILY_COLORS,
    size=5,
    alpha=0.75,
    jitter=0.22,
    ax=ax,
)
ax.axhline(50, color="#777777", linestyle="--", linewidth=1)
ax.axhline(72, color="#777777", linestyle="--", linewidth=1)
ax.axhline(90, color="#777777", linestyle="--", linewidth=1)
ax.set_title("Parcel-Level Preferred SRM+LLM Layer Depth")
ax.set_xlabel("Network")
ax.set_ylabel("Best SRM layer depth (%)")
ax.yaxis.set_major_formatter(FormatStrFormatter("%.0f"))
plt.xticks(rotation=35, ha="right")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title="Network family", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
savefig_clean(fig, FIG_DIR / "step11G_parcel_preferred_layer_depth_by_network.png")

# Figure 3: depth-bin proportions from Step 11E.
prop_cols = [f"{bin_name}_parcel_proportion" for bin_name in DEPTH_BIN_ORDER if f"{bin_name}_parcel_proportion" in network_final_df.columns]
if prop_cols:
    prop_df = network_final_df[["roi_name"] + prop_cols].copy()
    prop_df["roi_name"] = pd.Categorical(prop_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
    prop_df = prop_df.sort_values("roi_name")
    fig, ax = plt.subplots(figsize=(15, 6.8))
    bottom = np.zeros(len(prop_df))
    for bin_name in DEPTH_BIN_ORDER:
        col = f"{bin_name}_parcel_proportion"
        if col not in prop_df.columns:
            continue
        vals = prop_df[col].fillna(0).values
        ax.bar(prop_df["roi_name"].astype(str), vals, bottom=bottom, label=DEPTH_BIN_LABELS[bin_name], color=DEPTH_BIN_COLORS[bin_name], alpha=0.92)
        bottom += vals
    ax.set_title("Network Composition Of Preferred Layer-Depth Bins")
    ax.set_xlabel("Network")
    ax.set_ylabel("Parcel proportion")
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    plt.xticks(rotation=35, ha="right")
    ax.legend(title="Layer-depth bin", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    savefig_clean(fig, FIG_DIR / "step11G_network_layer_depth_bin_proportions.png")

# Figure 4: heterogeneity and spatial-gradient summary.
if {"heterogeneity_index", "best_spatial_model_adj_r2", "mean_best_srm_r"}.issubset(network_final_df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    sns.scatterplot(
        data=network_final_df,
        x="mean_best_srm_r",
        y="heterogeneity_index",
        hue="network_family",
        s=95,
        edgecolor="white",
        linewidth=0.8,
        palette=FAMILY_COLORS,
        ax=axes[0],
    )
    axes[0].set_title("Encoding Strength And Layer-Depth Heterogeneity")
    axes[0].set_xlabel("Mean best SRM+LLM encoding r")
    axes[0].set_ylabel("Within-network heterogeneity index")
    axes[0].xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    axes[0].yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    axes[0].legend_.remove()
    for _, row in network_final_df.iterrows():
        if np.isfinite(row.get("mean_best_srm_r", np.nan)) and np.isfinite(row.get("heterogeneity_index", np.nan)):
            axes[0].text(row["mean_best_srm_r"], row["heterogeneity_index"], str(row["roi_name"]), fontsize=8, ha="left", va="bottom")

    sns.scatterplot(
        data=network_final_df,
        x="mean_best_srm_r",
        y="best_spatial_model_adj_r2",
        hue="network_family",
        style="best_spatial_model_family",
        s=95,
        edgecolor="white",
        linewidth=0.8,
        palette=FAMILY_COLORS,
        ax=axes[1],
    )
    axes[1].set_title("Encoding Strength And Spatial Organization")
    axes[1].set_xlabel("Mean best SRM+LLM encoding r")
    axes[1].set_ylabel("Best spatial model adjusted R²")
    axes[1].xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    axes[1].yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    for _, row in network_final_df.iterrows():
        if np.isfinite(row.get("mean_best_srm_r", np.nan)) and np.isfinite(row.get("best_spatial_model_adj_r2", np.nan)):
            axes[1].text(row["mean_best_srm_r"], row["best_spatial_model_adj_r2"], str(row["roi_name"]), fontsize=8, ha="left", va="bottom")
    handles, labels = axes[1].get_legend_handles_labels()
    axes[1].legend(handles, labels, title="Network family / model type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    savefig_clean(fig, FIG_DIR / "step11G_heterogeneity_and_spatial_gradient_summary.png")

# Figure 5: parcel-level SRM gain versus preferred layer depth.
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=parcel_df,
    x="best_srm_log_depth_percent",
    y="same_layer_gain",
    hue="network_family",
    size="best_srm_r",
    sizes=(25, 180),
    alpha=0.78,
    edgecolor="white",
    linewidth=0.4,
    palette=FAMILY_COLORS,
    ax=ax,
)
ax.axhline(0, color="#333333", linewidth=1)
ax.axvline(50, color="#777777", linestyle="--", linewidth=1)
ax.axvline(72, color="#777777", linestyle="--", linewidth=1)
ax.axvline(90, color="#777777", linestyle="--", linewidth=1)
ax.set_title("Parcel SRM Gain Across Preferred Layer Depth")
ax.set_xlabel("Best SRM layer depth (%)")
ax.set_ylabel("SRM gain at preferred layer")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.legend(title="Network family / encoding r", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
savefig_clean(fig, FIG_DIR / "step11G_parcel_gain_by_preferred_layer_depth.png")

# Figure 6: statistical test summary, only compact display of strongest tests.
if len(stats_df):
    stat_plot_df = (
        stats_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["p_fdr_all_step11G_tests"])
        .copy()
        .sort_values("p_fdr_all_step11G_tests", ascending=True)
        .head(15)
    )
    if len(stat_plot_df):
        stat_plot_df["test_label"] = stat_plot_df["analysis"] + "\n" + stat_plot_df["metric"].astype(str).str.slice(0, 45)
        fig, ax = plt.subplots(figsize=(12, 8))
        pvals = stat_plot_df["p_fdr_all_step11G_tests"].clip(lower=1e-300)
        vals = -np.log10(pvals)
        ax.barh(stat_plot_df["test_label"], vals, color="#4C78A8", alpha=0.9)
        ax.axvline(-np.log10(0.05), color="#D95F02", linestyle="--", linewidth=1.5, label="FDR p = 0.05")
        ax.set_title("Strongest Parcel-Level Statistical Follow-Up Tests")
        ax.set_xlabel("-log10(FDR-adjusted p)")
        ax.set_ylabel("Test")
        ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.invert_yaxis()
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        plt.tight_layout()
        savefig_clean(fig, FIG_DIR / "step11G_statistical_followup_summary.png")

# =========================
# 5. Summary JSON and display
# =========================

total_elapsed = (time.time() - start_time) / 60.0
summary = {
    "step": "step11G_final_parcel_level_summary_figures_and_tables",
    "analysis_goal": "Integrate parcel-level layer preference, SRM gain, heterogeneity, and exploratory spatial-gradient outputs into final summary tables and figures.",
    "n_parcels": int(len(parcel_df)),
    "n_networks": int(parcel_df["roi_name"].nunique()),
    "n_statistical_tests": int(len(stats_df)),
    "primary_tables": {
        "final_parcel_table": str(final_parcel_csv),
        "final_network_summary": str(final_network_csv),
        "network_ranking_summary": str(network_ranking_csv),
        "statistical_tests": str(stats_csv),
        "top_parcels_by_best_srm_r": str(top_parcels_by_r_csv),
        "top_parcels_by_same_layer_gain": str(top_parcels_by_gain_csv),
    },
    "figure_dir": str(FIG_DIR),
    "notes": [
        "All p-values are formatted without scientific notation in display columns.",
        "Spatial-gradient results are exploratory and use the tightened Step 11F network-by-network model set.",
        "Legends are placed outside the right side of summary figures when applicable.",
    ],
    "elapsed_minutes": float(total_elapsed),
}

with open(JSON_DIR / "step11G_final_parcel_level_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11G completed: Final Parcel-Level Summary Figures And Tables")
print(f"Parcels: {len(parcel_df)}")
print(f"Networks: {parcel_df['roi_name'].nunique()}")
print(f"Statistical follow-up tests: {len(stats_df)}")
print(f"Saved final tables to: {CSV_DIR}")
print(f"Saved final figures to: {FIG_DIR}")
print(f"Elapsed: {total_elapsed:.2f} min")

print("\nTop networks by mean best SRM+LLM encoding r:")
show_cols = [c for c in ["roi_name", "network_family", "mean_best_srm_r", "mean_same_layer_gain", "mean_preferred_depth", "heterogeneity_index", "best_single_axis_name", "best_single_axis_fdr_p"] if c in network_ranking_df.columns]
print(network_ranking_df[show_cols].head(10).to_string(index=False))

if len(stats_df):
    print("\nStrongest statistical follow-up tests:")
    show_cols = ["analysis", "metric", "test", "statistic", "p_value_print", "p_fdr_all_step11G_tests_print", "significance_fdr"]
    print(stats_df.sort_values("p_fdr_all_step11G_tests")[show_cols].head(15).to_string(index=False))
